# Sesión 5 — Cassandra (I), curso 2026-27

![Logotipo de Apache Cassandra](https://raw.githubusercontent.com/dsevilla/bdge/26-27/docs/img/cassandra-logo-notebook.png)

Esta sesión presenta Cassandra desde Python: la conexión al clúster con el *driver*, el *keyspace* y las tablas creados a partir de los Parquet, la carga de los datos, las consultas por clave, los índices SAI y `ALLOW FILTERING`, la unión de tablas en el cliente, las colecciones, el nivel de consistencia y los contadores. La sesión 6 parte de esta misma base de datos.

El cuaderno funciona con un solo nodo (Google Colab o `docker-compose-cassandra.yml`) o con el clúster de dos nodos de `docker-compose-cassandra-2.yml`. Con Docker, arranca antes el Compose y abre el cuaderno en su servicio `notebook` (<http://localhost:8888>, token `bdge`), como explica `cassandra/README.md`: el cuaderno tiene que estar en la red de Compose para llegar a los nodos, y abierto desde fuera la conexión falla tras cinco minutos de reintentos.

In [ ]:
import sys

RunningInCOLAB: bool = "google.colab" in sys.modules

## Instalación de Apache Cassandra (sólo Google Colab)

In [ ]:
# Aparentemente han eliminado java de los paquetes instalados
if RunningInCOLAB:
    !sudo apt-get update && sudo DEBIAN_FRONTEND=noninteractive apt-get install -y default-jdk-headless --no-install-recommends
    %env JAVA_HOME=/usr/lib/jvm/default-java

Descargamos el ejecutable comprimido desde su página oficial, sólo si estamos en Colab. En las ejecuciones con Docker Compose y en las pruebas del CI, Cassandra ya se está ejecutando en otro contenedor o en otra máquina.

In [ ]:
if RunningInCOLAB:
    # Determinar qué versión 5.x.x de Cassandra es la más reciente.
    import re
    from http.client import HTTPResponse
    from re import Pattern
    from urllib.request import Request, urlopen

    from bs4 import BeautifulSoup

    version_re: Pattern = re.compile(r"^(5\.[0-9]+(\.[0-9]+)?)")
    req: Request = Request("https://dlcdn.apache.org/cassandra/")
    html_page: HTTPResponse = urlopen(req)

    soup: BeautifulSoup = BeautifulSoup(html_page)
    versions: list[str] = [
        m.group(0) for m in [version_re.match(link.get_text()) for link in soup.find_all("a")] if m
    ]
    if not versions:
        raise RuntimeError(
            "No se encontró ninguna versión 5.x en https://dlcdn.apache.org/cassandra/"
        )
    # Se comparan como números, no como cadenas: "5.0.10" es posterior a "5.0.9".
    bigger5x: str = max(versions, key=lambda version: tuple(int(p) for p in version.split(".")))
    %env CASSANDRA_VER=$bigger5x

Descarga de Cassandra y de su firma digital SHA256 para comprobar que el fichero es correcto. Si es correcto, se descomprime. Esto sólo se realiza en Google Colab. Cada descarga se reintenta con Tenacity (`@retry`): cuatro intentos con esperas de 2, 4 y 8 segundos y aviso antes de cada reintento, y sólo ante errores pasajeros (HTTP 5xx o 429, fallos de red). Si la CDN de Apache sigue fallando, se pide al archivo de Apache.

In [ ]:
if RunningInCOLAB:
    import hashlib
    import logging
    import os
    import shutil
    import tarfile
    from urllib.error import HTTPError, URLError
    from urllib.request import urlopen

    # Esta celda va antes de instalar las dependencias del cuaderno.
    %pip install -q tenacity
    from tenacity import (
        before_sleep_log,
        retry,
        retry_if_exception,
        stop_after_attempt,
        wait_exponential,
    )

    # La CDN de Apache (dlcdn) a veces responde con un error 5xx pasajero
    # («503: first byte timeout»). Cada descarga se reintenta con esperas
    # crecientes y, si la CDN sigue fallando, se pide al archivo de Apache, que
    # guarda todas las versiones. Un 404 u otro error del cliente no se reintenta.
    APACHE_MIRRORS: tuple[str, ...] = (
        "https://dlcdn.apache.org/cassandra",
        "https://archive.apache.org/dist/cassandra",
    )
    DOWNLOAD_ATTEMPTS: int = 4
    DOWNLOAD_RETRY_LOGGER: logging.Logger = logging.getLogger("download_retry")

    def download(path: str, filename: str) -> None:
        """Descarga `path` (relativo a la carpeta de Cassandra) en `filename`."""

        def is_transient(error: BaseException) -> bool:
            """Errores pasajeros: HTTP 5xx o 429 y fallos de red. Un 404 no lo es."""
            if isinstance(error, HTTPError):
                return error.code >= 500 or error.code == 429
            return isinstance(error, (URLError, TimeoutError, ConnectionError))

        # Cuatro intentos con esperas de 2, 4 y 8 s entre ellos.
        @retry(
            retry=retry_if_exception(is_transient),
            stop=stop_after_attempt(DOWNLOAD_ATTEMPTS),
            wait=wait_exponential(multiplier=2),
            before_sleep=before_sleep_log(DOWNLOAD_RETRY_LOGGER, logging.WARNING),
            reraise=True,
        )
        def download_from(url: str, filename: str) -> None:
            """Descarga `url` en `filename`."""
            with urlopen(url, timeout=60) as response, open(filename, "wb") as out:
                shutil.copyfileobj(response, out)

        for mirror in APACHE_MIRRORS:
            try:
                download_from(f"{mirror}/{path}", filename)
                return
            except OSError as error:
                if not is_transient(error):
                    raise
                failure: Exception = error
        raise RuntimeError(f"No se pudo descargar {path} de ningún servidor") from failure

    # Descargar Cassandra y su firma digital SHA256
    cassandra_version: str = os.getenv("CASSANDRA_VER", "5.0.0")
    tarball: str = f"apache-cassandra-{cassandra_version}-bin.tar.gz"
    download(f"{cassandra_version}/{tarball}", tarball)
    download(f"{cassandra_version}/{tarball}.sha256", f"{tarball}.sha256")

    # Comprobar la firma digital SHA256 (file_digest lee el fichero por bloques)
    with open(tarball, "rb") as f:
        actual_hash: str = hashlib.file_digest(f, "sha256").hexdigest()
    with open(f"{tarball}.sha256") as f:
        expected_hash: str = f.read().split()[0]
    if actual_hash != expected_hash:
        raise ValueError("Downloaded Cassandra binary does not match the expected SHA256 hash.")

    # Descomprimir Cassandra
    with tarfile.open(tarball, "r:gz") as tar:
        tar.extractall(filter="data")

Dependiendo de la versión de Java, hay que ejecutar Cassandra con un parámetro adicional.

In [ ]:
if RunningInCOLAB:
    javav = !javap -verbose java.lang.String | grep "major version" | cut -d ' ' -f 5
    cassandra_opts: str = ""
    if int(javav[0]) > 55:
        cassandra_opts = "-Djava.security.manager=allow"
    %env CASSANDRA_OPTS=$cassandra_opts

Usamos la configuración `cassandra_latest.yaml`, la que Cassandra incluye para instalaciones nuevas, con sus funciones más recientes activadas: índices SAI por omisión, entre otras. Cassandra sólo lee `cassandra.yaml`, así que la copiamos encima, igual que hacen los ficheros Compose.

Hay que cambiar además algunas opciones de esa configuración. Son límites de protección pensados para producción. El tamaño a partir del cual un *batch* genera un aviso o se rechaza se relaja, para que las cargas de la hoja no choquen con él:

In [ ]:
if RunningInCOLAB:
    import os
    import re
    import shutil

    conf_dir: str = f"apache-cassandra-{os.environ['CASSANDRA_VER']}/conf"
    config_file: str = f"{conf_dir}/cassandra.yaml"

    # Cassandra sólo lee cassandra.yaml: se copia encima cassandra_latest.yaml, la
    # configuración para instalaciones nuevas, y después se ajusta.
    shutil.copy(f"{conf_dir}/cassandra_latest.yaml", config_file)

    # Read the configuration file
    with open(config_file) as f:
        content: str = f.read()

    # Apply the same replacements as the sed command
    content = re.sub(r"(batch_size_warn_threshold:).*", r"\1 128KiB", content)
    content = re.sub(r"(batch_size_fail_threshold:).*", r"\1 1024KiB", content)

    # Write back the modified content
    with open(config_file, "w") as f:
        f.write(content)

Arrancamos la base de datos (el parámetro `-R` ejecuta el servidor en segundo plano y además permite ejecutarlo como `root`).

In [ ]:
if RunningInCOLAB:
    !(cd apache-cassandra-${CASSANDRA_VER}/ && sudo --preserve-env=MAX_HEAP_SIZE bin/cassandra -R ${CASSANDRA_OPTS})

Con el comando `nodetool status` podemos ver que nuestro clúster de Colab contiene un único *datacenter*, con un único nodo. Hacemos un bucle para esperar hasta que esté disponible, porque la inicialización lleva un tiempo.

In [ ]:
if RunningInCOLAB:
    import os
    import subprocess
    import time

    nodetool: str = f"apache-cassandra-{os.environ['CASSANDRA_VER']}/bin/nodetool"
    deadline: float = time.monotonic() + 600
    while True:
        status: str = subprocess.run(
            [nodetool, "-h", "localhost", "status"], capture_output=True, text=True
        ).stdout
        if any(line.startswith("UN ") for line in status.splitlines()):
            break
        if time.monotonic() > deadline:
            # Si el proceso murió (por ejemplo, sin memoria), el registro dice por qué.
            !tail -n 100 apache-cassandra-${CASSANDRA_VER}/logs/system.log
            raise TimeoutError("Cassandra no arrancó en 10 minutos.")
        time.sleep(5)
    print("Cassandra is up and running.")

Si quisiéramos abrir el shell de comandos de Cassandra para ejecutar instrucciones directamente sobre ella deberíamos ejecutar el siguiente comando.

*De todas formas, dejamos la celda comentada puesto que nuestro interés es en acceder a la BD mediante Python*

In [ ]:
#!apache-cassandra-${CASSANDRA_VER}/bin/cqlsh

## Acceso a Cassandra mediante Python

Instalamos la librería `cassandra-driver` que nos dará acceso a nuestra instancia de Cassandra.

In [ ]:
%pip install cassandra-driver fsspec pyarrow aiohttp tenacity

In [ ]:
# @title Diapositiva: El recorrido de Cassandra
from IPython.display import HTML, display; display(HTML('<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#fff9db;color:#29261d;border:1px solid #dfcc78;border-top:10px solid #c69200;border-radius:10px;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:22px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#c69200" size="8" noshade style="display:none"><div style="color:#c69200;font-size:15px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#c69200" style="color:inherit"><b>A continuación</b></font></div><h1 style="color:#7b2c19;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:34px;line-height:1.15;margin:0 0 22px">El recorrido de Cassandra</h1><ul style=\'margin:18px 0 8px;padding-left:30px\'><li style=\'margin:10px 0;list-style-position:outside\'>Conectar con un clúster y descubrir nodos, perfiles y <em>datacenters</em></li><li style=\'margin:10px 0;list-style-position:outside\'>Diseñar tablas según consultas y claves de partición y <em>clustering</em></li><li style=\'margin:10px 0;list-style-position:outside\'>Cargar Parquet por lotes con <em>statements</em> preparados</li><li style=\'margin:10px 0;list-style-position:outside\'>Contrastar índices SAI, <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>ALLOW FILTERING</code> y desnormalización</li></ul><blockquote style=\'background:#fff0ae;color:inherit;border:none;border-left:6px solid #c69200;margin:22px 0 0;padding:12px 16px\'><div style=\'margin-top:0px\'>Idea central: en Cassandra modelamos la consulta antes que la tabla</div></blockquote><hr color="#c69200" size="2" noshade style="display:none"></div>'))

In [ ]:
# @title Diapositiva: Entrar en un clúster Cassandra
from IPython.display import HTML, display; display(HTML('<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#fff9db;color:#29261d;border:1px solid #dfcc78;border-top:10px solid #c69200;border-radius:10px;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:22px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#c69200" size="8" noshade style="display:none"><div style="color:#c69200;font-size:15px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#c69200" style="color:inherit"><b>A continuación</b></font></div><h1 style="color:#7b2c19;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:34px;line-height:1.15;margin:0 0 22px">Entrar en un clúster Cassandra</h1><ul style=\'margin:18px 0 8px;padding-left:30px\'><li style=\'margin:10px 0;list-style-position:outside\'>Usar <em>contact points</em> sólo para iniciar el descubrimiento</li><li style=\'margin:10px 0;list-style-position:outside\'>Dejar que el <em>driver</em> conozca todos los nodos accesibles</li><li style=\'margin:10px 0;list-style-position:outside\'>Crear perfiles para consistencia, <em>timeouts</em> y encaminamiento</li><li style=\'margin:10px 0;list-style-position:outside\'>Elegir el <em>datacenter</em> a partir de los metadatos, no de un nombre supuesto</li></ul><blockquote style=\'background:#fff0ae;color:inherit;border:none;border-left:6px solid #c69200;margin:22px 0 0;padding:12px 16px\'><div style=\'margin-top:0px\'>Conectar es descubrir la topología y decidir cómo viajará cada consulta</div></blockquote><hr color="#c69200" size="2" noshade style="display:none"></div>'))

## Conexión al clúster: puntos de contacto y perfiles de ejecución

Conectarse a Cassandra no se parece a abrir una conexión contra un servidor SQL. El *driver* mantiene conexiones **con todos los nodos que considera accesibles** y decide, consulta a consulta, a cuál se la envía. Conviene separar tres conceptos:

1. **Puntos de contacto** (`contact_points`): sólo se usan en el arranque. El *driver* se conecta a uno de ellos, lee `system.local` y `system.peers` (tablas del sistema) y descubre el resto del clúster. Basta con uno accesible; se indican varios por tolerancia a fallos en el arranque.
2. **Hosts** (`cassandra.pool.Host`): cada nodo descubierto, con su dirección, su *datacenter*, su *rack* y su estado. Se listan con `cluster.metadata.all_hosts()`. El *driver* se conecta a cada nodo por la dirección que **el propio nodo publica** (`rpc_address` / `broadcast_rpc_address`), no por la que escribimos en `contact_points`.
3. **Perfiles de ejecución** (`ExecutionProfile`): agrupan las decisiones que afectan a una consulta —política de balanceo (qué nodo la atiende), nivel de consistencia, *timeout* y formato de las filas (`row_factory`)—. Cada consulta elige uno con `session.execute(..., execution_profile="nombre")`.

![El driver arranca con un único punto de contacto, descubre el resto de nodos con system.local y system.peers y abre conexiones con cada uno en la dirección que publica](https://raw.githubusercontent.com/dsevilla/bdge/26-27/docs/img/fig-cassandraconnect.svg)


<details><summary>Políticas de balanceo <span style="color: #007acc;">▶️ Haz clic para verlas</span></summary>

- `RoundRobinPolicy`: reparte entre todos los nodos vivos.
- `DCAwareRoundRobinPolicy(local_dc=...)`: sólo usa los nodos del *datacenter* local. Si `local_dc` no coincide **exactamente** con el nombre real del DC, todos los nodos se consideran remotos y el *driver* falla con `NoHostAvailable`.
- `TokenAwarePolicy(politica_hija)`: envuelve a otra política y, cuando conoce la clave de partición (con sentencias preparadas), envía la petición directamente a una **réplica** de esa partición, ahorrando un salto de coordinación.
- `WhiteListRoundRobinPolicy([direcciones])` y `HostFilterPolicy(politica_hija, predicate)`: restringen los nodos utilizables. Sirven para administración o depuración, no para producción: anulan el reparto de carga y el *token awareness*.

![Con cuatro nodos en dos datacenters, RoundRobinPolicy usa todos por turno, TokenAwarePolicy sólo las réplicas de la partición, DCAwareRoundRobinPolicy sólo los del DC local y WhiteListRoundRobinPolicy sólo los de la lista](https://raw.githubusercontent.com/dsevilla/bdge/26-27/docs/img/fig-cassandrapolicies.svg)

</details>

Definimos dos perfiles: el de consultas interactivas (el perfil por omisión) y otro para la carga masiva de datos, que puede usar un nivel de consistencia más débil y un *timeout* mayor.

In [ ]:
# @title Diapositiva: Cuántas réplicas deben responder
from IPython.display import HTML, display; display(HTML('<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#fff9db;color:#29261d;border:1px solid #dfcc78;border-top:10px solid #c69200;border-radius:10px;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:22px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#c69200" size="8" noshade style="display:none"><div style="color:#c69200;font-size:15px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#c69200" style="color:inherit"><b>A continuación</b></font></div><h1 style="color:#7b2c19;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:34px;line-height:1.15;margin:0 0 22px">Cuántas réplicas deben responder</h1><ul style=\'margin:18px 0 8px;padding-left:30px\'><li style=\'margin:10px 0;list-style-position:outside\'><code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>ONE</code>: basta una réplica; rápido y disponible, pero puede devolver un dato atrasado</li><li style=\'margin:10px 0;list-style-position:outside\'><code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>QUORUM</code>: la mayoría, <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>n // 2 + 1</code>; reconcilia réplicas sin exigir que estén todas</li><li style=\'margin:10px 0;list-style-position:outside\'><code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>ALL</code>: las <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>n</code> réplicas; falla en cuanto cae un nodo</li><li style=\'margin:10px 0;list-style-position:outside\'>Si <strong>R + W &gt; RF</strong>, toda lectura comparte al menos una réplica con la última escritura</li><li style=\'margin:10px 0;list-style-position:outside\'>El perfil por omisión usa <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>LOCAL_QUORUM</code>; la carga masiva, <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>ANY</code></li></ul><blockquote style=\'background:#fff0ae;color:inherit;border:none;border-left:6px solid #c69200;margin:22px 0 0;padding:12px 16px\'><div style=\'margin-top:0px\'>El precio de la consistencia es la disponibilidad</div></blockquote><hr color="#c69200" size="2" noshade style="display:none"></div>'))

### Nivel de consistencia: opciones y cuál usamos

Cada perfil fija también el **nivel de consistencia**: cuántas réplicas de una partición deben responder para dar por buena una lectura o una escritura. Con factor de replicación (RF) `n`, las opciones principales son:

| Nivel | Réplicas que deben responder | Compromiso |
|---|---|---|
| `ANY` (sólo escritura) | ninguna: basta con que el coordinador guarde una *pista* (*hint*) si no hay réplica disponible | la escritura más rápida y disponible, pero puede tardar en ser visible |
| `ONE` / `LOCAL_ONE` | 1 | rápido y muy disponible, pero puede leer una réplica que aún no ha recibido la última escritura |
| `QUORUM` / `LOCAL_QUORUM` | la mayoría, `n // 2 + 1` | equilibrio: consistencia sin exigir que estén todas vivas |
| `ALL` | las `n` | la máxima consistencia, pero falla en cuanto cae un solo nodo |

El prefijo `LOCAL_` cuenta sólo las réplicas del *datacenter* local; con un único DC, como aquí, equivale al nivel sin prefijo.

Una escritura se envía a todas las réplicas, pero el coordinador responde en cuanto contestan las que exige el nivel; las demás la aplican un instante *después*. Por eso una lectura a `ONE` justo después de una escritura a `ONE` puede caer en una réplica atrasada y devolver el dato antiguo. Una lectura de varias réplicas, en cambio, compara sus versiones y se queda con la de marca temporal más reciente. La regla que garantiza ver la última escritura es **R + W > RF**: si las réplicas leídas (R) y las escritas (W) suman más que RF, siempre comparten al menos una.

![Con tres réplicas, toda lectura QUORUM tras una escritura QUORUM incluye una réplica con el valor nuevo; tras una escritura ONE, dos de las tres lecturas ONE posibles devuelven el valor antiguo](https://raw.githubusercontent.com/dsevilla/bdge/26-27/docs/img/fig-consistencyoverlap.svg)

Por eso el perfil por omisión usa **`LOCAL_QUORUM`**. En el clúster de dos nodos (RF = 2) la mayoría es 2, así que R + W = 4 > 2 y cada lectura ve lo último que se escribió: las comprobaciones de los ejercicios, que leen enseguida lo que se acaba de escribir, no dependen de qué réplica responda antes. El EJERCICIO 5, más abajo, calcula estas cantidades para RF = 3. El precio es la disponibilidad: con RF = 2, si cae un nodo la operación falla con `Unavailable`, mientras que `ONE` seguiría respondiendo. En producción se usa RF = 3, donde la mayoría (2) sigue disponible con un nodo caído. Con un solo nodo (RF = 1) la mayoría es 1 y se comporta como `ONE`.

La carga masiva es distinta: el perfil de carga usa `ANY`, porque sólo importa la velocidad y nadie lee a mitad de la carga. Las lecturas posteriores, ya con `LOCAL_QUORUM`, consultan las dos réplicas y reconcilian.

In [ ]:
import logging
import os

from cassandra.cluster import (
    EXEC_PROFILE_DEFAULT,
    Cluster,
    ExecutionProfile,
    NoHostAvailable,
    Session,
)
from cassandra.policies import RoundRobinPolicy, TokenAwarePolicy
from cassandra.query import named_tuple_factory
from tenacity import (
    before_sleep_log,
    retry,
    retry_if_exception_type,
    stop_after_delay,
    wait_exponential,
)

from cassandra import ConsistencyLevel

# Puntos de contacto: en Colab, el nodo local; con Docker, el contenido de
# DB_HOSTNAME (que puede llevar varios nombres separados por comas). Sólo se
# usan para arrancar: los demás nodos se descubren leyendo `system.peers`.
contact_points: list[str] = (
    ["localhost"]
    if RunningInCOLAB
    else [name.strip() for name in os.getenv("DB_HOSTNAME", "localhost").split(",") if name.strip()]
)

# Nombre del perfil que usaremos para la carga masiva de datos.
LOAD_PROFILE: str = "load"

# Perfil por omisión: consultas interactivas de la hoja. LOCAL_QUORUM: lecturas y
# escrituras contactan con la mayoría de réplicas y las lecturas reconcilian
# (véase la explicación anterior); con un solo nodo equivale a ONE.
query_profile: ExecutionProfile = ExecutionProfile(
    load_balancing_policy=TokenAwarePolicy(RoundRobinPolicy()),
    consistency_level=ConsistencyLevel.LOCAL_QUORUM,
    request_timeout=40.0,
    row_factory=named_tuple_factory,
)

# Perfil de carga: las escrituras masivas admiten el nivel más débil (ANY, que
# sólo existe para escritura) y necesitan más tiempo por petición. Las lecturas
# posteriores usan el perfil por omisión, que reconcilia las réplicas.
load_profile: ExecutionProfile = ExecutionProfile(
    load_balancing_policy=TokenAwarePolicy(RoundRobinPolicy()),
    consistency_level=ConsistencyLevel.ANY,
    request_timeout=120.0,
    row_factory=named_tuple_factory,
)

cluster: Cluster = Cluster(
    contact_points,
    port=9042,
    execution_profiles={EXEC_PROFILE_DEFAULT: query_profile, LOAD_PROFILE: load_profile},
    # Tras cada CREATE, ALTER o DROP, el driver espera a que todos los nodos tengan
    # la misma versión del esquema antes de devolver el control. Por omisión espera
    # 10 s y, si se agota, sigue sin error: en el clúster de dos nodos no siempre
    # bastaba, y la escritura siguiente llegaba a un nodo que aún no conocía la
    # tabla recién creada (INCOMPATIBLE_SCHEMA).
    max_schema_agreement_wait=30,
)

# Los contenedores pueden tardar en aceptar conexiones, así que `connect()` se
# reintenta con *exponential backoff*: cada intento espera el doble que el
# anterior (1, 2, 4... segundos, con un tope de 30). Si el nodo sólo tardaba un
# momento, se reintenta enseguida; si tarda en arrancar, no se le satura con
# intentos cada pocos segundos. Lo que acota la espera es un plazo total, no un
# número de intentos. Sólo se reintenta `NoHostAvailable`: cualquier otro error
# es un fallo de configuración y debe verse en el acto.
CONNECT_TIMEOUT_S: float = 300.0
CONNECT_RETRY_LOGGER: logging.Logger = logging.getLogger("cassandra_connect_retry")


@retry(
    retry=retry_if_exception_type(NoHostAvailable),
    stop=stop_after_delay(CONNECT_TIMEOUT_S),
    wait=wait_exponential(multiplier=1, max=30),
    before_sleep=before_sleep_log(CONNECT_RETRY_LOGGER, logging.WARNING),
    reraise=True,
)
def connect_to_cluster(cluster: Cluster) -> Session:
    """Abre la sesión del clúster, esperando a que éste acepte conexiones."""
    return cluster.connect()


# Establecer la variable de sesión. Será nuestra conexión a Cassandra.
# Ojo: NO se debe hacer `session.default_timeout = ...` estando en modo de
# perfiles; ese valor es ahora `request_timeout` dentro de cada perfil.
session: Session = connect_to_cluster(cluster)

release_version: str = session.execute("SELECT release_version FROM system.local").one()[0]

print(f"Puntos de contacto: {contact_points}")
print(f"Versión de Cassandra: {release_version}")

Los contenedores pueden tardar en aceptar conexiones, así que `cluster.connect()` va dentro de una función decorada con `@retry` de Tenacity: si el clúster todavía no responde (`NoHostAvailable`), repite el intento con esperas crecientes (1, 2, 4... hasta 30 segundos) durante un máximo de cinco minutos, avisando antes de cada reintento. Cualquier otro error se propaga en el acto.

Una vez conectados, el *driver* ya conoce el clúster completo, aunque le hayamos dado un único punto de contacto. Listamos los nodos con sus metadatos:

In [ ]:
from cassandra.pool import Host

hosts: list[Host] = sorted(cluster.metadata.all_hosts(), key=lambda host: str(host.address))
nhosts: int = len(hosts)

for host in hosts:
    print(
        f"{host.address:<16} dc={host.datacenter} rack={host.rack} "
        f"versión={host.release_version} activo={host.is_up}"
    )
nhosts

### Otros perfiles (consulta opcional)

En esta práctica sólo usamos los perfiles por omisión (`LOCAL_QUORUM`) y de carga (`ANY`), definidos al crear el clúster. El *driver* permite añadir otros con `cluster.add_execution_profile()`; no hacen falta para los ejercicios.

<details><summary>Dirigir una consulta a un nodo o a un <em>datacenter</em></summary>

Para administración o depuración se puede dirigir una consulta a un nodo concreto. Esto anula el reparto de carga y el *token awareness*, por lo que no es un patrón de producción. La dirección se obtiene del nodo descubierto, no del nombre del servicio de Docker:

```python
from cassandra.policies import WhiteListRoundRobinPolicy

host = sorted(cluster.metadata.all_hosts(), key=lambda host: str(host.address))[0]
cluster.add_execution_profile(
    "node1",
    ExecutionProfile(
        load_balancing_policy=WhiteListRoundRobinPolicy([host.address]),
        consistency_level=ConsistencyLevel.ONE,
        request_timeout=40.0,
        row_factory=named_tuple_factory,
    ),
)
session.execute("SELECT host_id FROM system.local", execution_profile="node1").one()
```

En varios *datacenters*, la política local evita enviar consultas por la WAN. El nombre del DC se toma de los metadatos; uno erróneo puede dejar el clúster sin nodos utilizables:

```python
from cassandra.policies import DCAwareRoundRobinPolicy

cluster.add_execution_profile(
    "local_dc",
    ExecutionProfile(
        load_balancing_policy=TokenAwarePolicy(
            DCAwareRoundRobinPolicy(local_dc=host.datacenter)
        ),
        consistency_level=ConsistencyLevel.LOCAL_ONE,
        request_timeout=40.0,
        row_factory=named_tuple_factory,
    ),
)
```

</details>

![Cada llamada a session.execute elige un perfil: el de omisión con LOCAL_QUORUM y el de carga con ANY; los dos se pasan al crear el clúster](https://raw.githubusercontent.com/dsevilla/bdge/26-27/docs/img/fig-cassandraprofilesbasic.svg)


### El clúster simulado con contenedores

`docker-compose-cassandra.yml` levanta un nodo y `docker-compose-cassandra-2.yml`, dos (`cassandra1` y `cassandra2`), en la red `cassandra-net`. Cada nodo publica la dirección de su interfaz **dentro de la red de Docker**, así que el cuaderno debe ejecutarse en el servicio `notebook` de ese mismo `docker-compose`. Desde el anfitrión, el *driver* descubriría al segundo nodo con una dirección `172.x.y.z` inalcanzable y daría *timeouts* intermitentes; desde allí sólo funciona `cqlsh` (`cqlsh localhost 9042`), que habla con un único nodo.

In [ ]:
# @title Diapositiva: Conectar no es sólo abrir un socket
from IPython.display import HTML, display; display(HTML('<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#fff9db;color:#29261d;border:1px solid #dfcc78;border-top:10px solid #287681;border-radius:10px;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:22px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#287681" size="8" noshade style="display:none"><div style="color:#287681;font-size:15px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#287681" style="color:inherit"><b>Recapitulación</b></font></div><h1 style="color:#7b2c19;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:34px;line-height:1.15;margin:0 0 22px">Conectar no es sólo abrir un socket</h1><ul style=\'margin:18px 0 8px;padding-left:30px\'><li style=\'margin:10px 0;list-style-position:outside\'>El <em>driver</em> mantiene conexiones con los nodos que descubre</li><li style=\'margin:10px 0;list-style-position:outside\'>Los perfiles separan la política de carga de la consulta concreta</li><li style=\'margin:10px 0;list-style-position:outside\'>El <em>datacenter</em> local evita tratar todos los nodos como remotos</li><li style=\'margin:10px 0;list-style-position:outside\'>El clúster de Docker permite ver la diferencia entre uno y varios nodos</li></ul><blockquote style=\'background:#fff0ae;color:inherit;border:none;border-left:6px solid #287681;margin:22px 0 0;padding:12px 16px\'><div style=\'margin-top:0px\'>Ya conocemos dónde se ejecuta una petición; ahora diseñaremos qué escribir</div></blockquote><hr color="#287681" size="2" noshade style="display:none"></div>'))

In [ ]:
# @title Diapositiva: Del esquema Parquet al CQL
from IPython.display import HTML, display; display(HTML('<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#fff9db;color:#29261d;border:1px solid #dfcc78;border-top:10px solid #c69200;border-radius:10px;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:22px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#c69200" size="8" noshade style="display:none"><div style="color:#c69200;font-size:15px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#c69200" style="color:inherit"><b>A continuación</b></font></div><h1 style="color:#7b2c19;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:34px;line-height:1.15;margin:0 0 22px">Del esquema Parquet al CQL</h1><ul style=\'margin:18px 0 8px;padding-left:30px\'><li style=\'margin:10px 0;list-style-position:outside\'>Leer el <em>footer</em> remoto sin descargar el fichero completo</li><li style=\'margin:10px 0;list-style-position:outside\'>Convertir metadatos de claves y nulabilidad en tipos CQL</li><li style=\'margin:10px 0;list-style-position:outside\'>Crear <em>keyspace</em> y tablas con una clave de partición explícita</li><li style=\'margin:10px 0;list-style-position:outside\'>Preparar la carga por lotes para las consultas que vendrán después</li></ul><blockquote style=\'background:#fff0ae;color:inherit;border:none;border-left:6px solid #c69200;margin:22px 0 0;padding:12px 16px\'><div style=\'margin-top:0px\'>En Cassandra el esquema nace de las preguntas que queremos contestar</div></blockquote><hr color="#c69200" size="2" noshade style="display:none"></div>'))

## Datos: esquemas y carga por lotes

Los datos de *stackoverflow* están publicados como ficheros Parquet en una *release* de GitHub. **No los descargamos enteros**: primero leemos sólo su esquema (el *footer* del Parquet, unos pocos KiB) para generar el CQL de creación de tablas, y después los recorremos por lotes (`RecordBatch`) mientras insertamos, de forma que la memoria ocupada no dependa del tamaño del fichero.

In [ ]:
# @title Diapositiva: Leer el esquema antes de repartir datos
from IPython.display import HTML, display; display(HTML('<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#fff9db;color:#29261d;border:1px solid #dfcc78;border-top:10px solid #c69200;border-radius:10px;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:22px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#c69200" size="8" noshade style="display:none"><div style="color:#c69200;font-size:15px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#c69200" style="color:inherit"><b>A continuación</b></font></div><h1 style="color:#7b2c19;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:34px;line-height:1.15;margin:0 0 22px">Leer el esquema antes de repartir datos</h1><ul style=\'margin:18px 0 8px;padding-left:30px\'><li style=\'margin:10px 0;list-style-position:outside\'>El <em>dump</em> contiene usuarios, publicaciones, comentarios, votos y etiquetas</li><li style=\'margin:10px 0;list-style-position:outside\'><code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>Id</code> y las referencias ayudan a decidir claves de partición y tablas derivadas</li><li style=\'margin:10px 0;list-style-position:outside\'>Los valores ausentes deben convertirse con una política explícita</li><li style=\'margin:10px 0;list-style-position:outside\'>El mismo Parquet alimentará varias tablas orientadas a consultas distintas</li></ul><blockquote style=\'background:#fff0ae;color:inherit;border:none;border-left:6px solid #c69200;margin:22px 0 0;padding:12px 16px\'><div style=\'margin-top:0px\'>En Cassandra el esquema de entrada es una pista, no una tabla final</div></blockquote><hr color="#c69200" size="2" noshade style="display:none"></div>'))

<details><summary><b>🗂️ Formato de los datos de Stack Overflow</b> <span style="color: #007acc;">▶️ Haz clic para ver el esquema completo</span></summary>

La descripción siguiente resume la [documentación vigente del *data dump* y de SEDE](https://meta.stackexchange.com/questions/2677/database-schema-documentation-for-the-public-data-dump-and-sede/2678). Las prácticas utilizan el *dump* de Stack Overflow en español; SEDE contiene además tablas que no se distribuyen en esos ocho ficheros XML.

> **Notas generales**
>
> - Cada fichero contiene elementos `<row ... />` codificados en UTF-8 y se distribuye dentro de un archivo 7z.
> - Los campos opcionales cuyo valor es nulo no aparecen como atributos en el XML. En los Parquet de la asignatura se representan normalmente como `null`.
> - Las fechas están en UTC y suelen usar el formato `YYYY-MM-DDTHH:mm:ss.fff`.
> - `Id` identifica la fila; los campos indicados como FK hacen referencia al `Id` de otra tabla.

---

:file_folder: <b>Badges.xml</b> — insignias concedidas a usuarios

- `Id`: clave primaria de la concesión.
- `UserId`: usuario que recibe la insignia; FK a `Users.Id`.
- `Name`: nombre de la insignia.
- `Date`: fecha de concesión.
- `Class`: clase de insignia (`1` = oro, `2` = plata, `3` = bronce).
- `TagBased`: `True` si está asociada a una etiqueta; `False` si es una insignia con nombre propio.

---

:file_folder: <b>Comments.xml</b> — comentarios en publicaciones

- `Id`: clave primaria.
- `PostId`: publicación comentada; FK a `Posts.Id`.
- `Score`: puntuación del comentario.
- `Text`: cuerpo del comentario.
- `CreationDate`: fecha de creación.
- `UserDisplayName`: nombre visible del autor cuando no puede referenciarse mediante `UserId`.
- `UserId`: autor; FK opcional a `Users.Id`, ausente si el usuario fue eliminado.
- `ContentLicense`: licencia Creative Commons del contenido.

---

:file_folder: <b>Posts.xml</b> — publicaciones no eliminadas

- `Id`: clave primaria.
- `PostTypeId`: tipo de publicación:
  - `1` pregunta; `2` respuesta; `3` wiki de etiqueta huérfana; `4` extracto de wiki de etiqueta; `5` wiki de etiqueta.
  - `6` candidatura a moderación; `7` marcador wiki; `8` wiki de privilegio; `9` artículo; `10` artículo de ayuda.
  - `12` colección; `13` respuesta a cuestionario de moderación; `14` anuncio; `15` discusión de colectivo; `17` colección de colectivo.
- `AcceptedAnswerId`: respuesta aceptada; FK opcional a `Posts.Id`, sólo para preguntas.
- `ParentId`: pregunta a la que responde; FK a `Posts.Id`, sólo para respuestas.
- `CreationDate`: fecha de creación.
- `Score`: puntuación neta.
- `ViewCount`: número de visitas, opcional.
- `Body`: cuerpo ya renderizado como HTML, no Markdown.
- `OwnerUserId`: autor; FK opcional a `Users.Id`. No aparece si se eliminó el usuario y vale `-1` en las entradas de wiki de etiqueta.
- `OwnerDisplayName`: nombre del autor cuando no hay un usuario referenciable.
- `LastEditorUserId`: último editor; FK opcional a `Users.Id`.
- `LastEditorDisplayName`: nombre del último editor cuando no hay un usuario referenciable.
- `LastEditDate`: fecha de la edición más reciente, opcional.
- `LastActivityDate`: fecha de la actividad más reciente.
- `Title`: título de la pregunta o, en determinados posts de tipo `4`/`5`, nombre de la etiqueta.
- `Tags`: etiquetas de la pregunta o etiqueta asociada a determinados posts de tipo `4`/`5`.
- `AnswerCount`: número de respuestas no eliminadas; sólo aparece en preguntas.
- `CommentCount`: número de comentarios, opcional.
- `FavoriteCount`: número histórico de favoritos, opcional.
- `ClosedDate`: fecha de cierre, sólo si la publicación está cerrada.
- `CommunityOwnedDate`: fecha desde la que la publicación es *community wiki*, si corresponde.
- `ContentLicense`: licencia Creative Commons del contenido.

Desde el *dump* de julio de 2025, `Posts.xml` incluye dos publicaciones artificiales con los identificadores `1000000001` y `1000000010`; no existen en SEDE y pueden excluirse. `DeletionDate` pertenece a `PostsWithDeleted` en SEDE y **no** aparece en el *data dump*.

---

:file_folder: <b>PostHistory.xml</b> — historial de las publicaciones

- `Id`: clave primaria del evento.
- `PostHistoryTypeId`: tipo de evento:
  - `1` título inicial; `2` cuerpo inicial; `3` etiquetas iniciales.
  - `4` edición de título; `5` edición de cuerpo; `6` edición de etiquetas.
  - `7`, `8`, `9`: reversión de título, cuerpo o etiquetas.
  - `10` cierre; `11` reapertura; `12` eliminación; `13` restauración; `14` bloqueo; `15` desbloqueo; `16` pasa a *community wiki*.
  - `17` migración antigua (sustituida por `35`/`36`); `18` fusión; `19` protección; `20` desprotección; `21` desasociación del autor; `22` deshacer fusión.
  - `24` edición sugerida aplicada; `25` publicación tuiteada; `31` discusión de comentarios movida al chat.
  - `33` aviso añadido; `34` aviso retirado; `35` migrada a otro sitio; `36` migrada desde otro sitio; `37` origen de fusión; `38` destino de fusión.
  - `50` subida por el usuario Community; `52` entrada en Hot Network Questions; `53` retirada de HNQ por moderación; `66` creada mediante el asistente de preguntas.
- `PostId`: publicación afectada; FK a `Posts.Id`.
- `RevisionGUID`: agrupa los registros producidos por una misma revisión.
- `CreationDate`: fecha del evento.
- `UserId`: usuario que realizó el cambio; FK opcional a `Users.Id`.
- `UserDisplayName`: nombre visible si el usuario fue eliminado o si se trata del autor de una publicación migrada.
- `Comment`: comentario de edición; para el tipo `10` contiene el motivo de cierre y para `33`/`34`, el identificador del aviso.
- `Text`: valor nuevo en crudo. En ciertos eventos de moderación contiene JSON con los votantes; en duplicados incluye `OriginalQuestionIds`, y para el tipo `17` contiene los detalles de migración.
- `ContentLicense`: licencia Creative Commons del contenido.

---

:file_folder: <b>PostLinks.xml</b> — enlaces entre publicaciones

- `Id`: clave primaria.
- `CreationDate`: fecha de creación del enlace.
- `PostId`: publicación origen; FK a `Posts.Id`.
- `RelatedPostId`: publicación relacionada o destino; FK a `Posts.Id`.
- `LinkTypeId`: tipo de enlace (`1` = enlazada, `3` = duplicada).

---

:file_folder: <b>Users.xml</b> — usuarios

- `Id`: clave primaria.
- `Reputation`: reputación.
- `CreationDate`: fecha de alta.
- `DisplayName`: nombre visible.
- `LastAccessDate`: última carga de una página por el usuario; se actualiza como máximo cada 30 minutos.
- `WebsiteUrl`: sitio web, opcional.
- `Location`: ubicación declarada, opcional.
- `AboutMe`: presentación del perfil, opcional.
- `Views`: visitas al perfil.
- `UpVotes`: votos positivos emitidos.
- `DownVotes`: votos negativos emitidos.
- `ProfileImageUrl`: URL de la imagen de perfil, opcional.
- `EmailHash`: actualmente siempre es nulo y, por tanto, no aparece en el XML.
- `AccountId`: identificador del perfil de red de Stack Exchange; opcional si el usuario oculta esa comunidad en su perfil.

El antiguo campo `Age` ya no forma parte del esquema documentado.

---

:file_folder: <b>Votes.xml</b> — votos y otras acciones sobre publicaciones

- `Id`: clave primaria.
- `PostId`: publicación afectada; FK a `Posts.Id`.
- `VoteTypeId`: tipo de voto o acción:
  - `-1` informar a moderación; `0` deshacer voto de moderación; `1` aceptación por el autor; `2` voto positivo; `3` voto negativo; `4` ofensivo.
  - `5` favorito/marcador (función retirada en octubre de 2022); `6` cierre (desde 2013 se guarda sólo en `PostHistory`); `7` reapertura.
  - `8` inicio de recompensa; `9` fin de recompensa; `10` eliminación; `11` restauración; `12` spam.
  - `15` revisión de moderación; `16` aprobación de edición sugerida; `17`–`21` reacciones/seguimiento; `22`–`28` reacciones de Teams.
  - `29` obsoleta; `30` no obsoleta; `31` voto previo; `32` voto positivo y `33` voto negativo en discusión de colectivo; `35`–`37` evaluación privada de respuesta de IA.
- `UserId`: usuario; sólo está presente para los tipos `5` y `8` (`-1` si se eliminó).
- `CreationDate`: fecha del voto. Por privacidad, la hora se elimina.
- `BountyAmount`: importe de la recompensa; sólo para los tipos `8` y `9`.

No todos los tipos enumerados pueden aparecer en SEDE o en el *dump* público; algunos son exclusivos de Teams o de experimentos concretos.

---

:file_folder: <b>Tags.xml</b> — etiquetas

- `Id`: clave primaria.
- `TagName`: nombre de la etiqueta.
- `Count`: número de preguntas que la usan.
- `ExcerptPostId`: publicación con el extracto de la wiki; FK opcional a `Posts.Id`.
- `WikiPostId`: publicación con el texto de la wiki; FK opcional a `Posts.Id`.
- `IsModeratorOnly`: indica si sólo moderación puede usarla.
- `IsRequired`: indica si su uso es obligatorio.

---

### Diagrama entidad-relación de SEDE

![Diagrama entidad-relación de Stack Exchange Data Explorer](https://raw.githubusercontent.com/dsevilla/bdge/26-27/docs/img/stackexchange-sede-erd.png)

El diagrama corresponde a **SEDE**, cuyo esquema es más rico que los ocho ficheros del *data dump*. Se incluye como visión global de tablas y relaciones; no implica que todas esas tablas estén disponibles en los Parquet de las prácticas. Fuente y autoría: [“The Schema Visualized”](https://meta.stackexchange.com/a/326361).

</details>

In [ ]:
from collections.abc import Iterator

import fsspec
import pyarrow as pa
import pyarrow.parquet as pq

PARQUET_BASE_URL: str = (
    "https://github.com/dsevilla/bd2-data/releases/download/es.stackoverflow.data-26-27"
)


def read_remote_parquet_schema(url: str) -> pa.Schema:
    """Lee el esquema Arrow pidiendo por HTTP sólo el *footer* del Parquet."""
    with fsspec.open(url, "rb") as parquet_file:
        return pq.read_schema(parquet_file)


def iter_remote_parquet_batches(url: str, batch_size: int = 20_000) -> Iterator[pa.RecordBatch]:
    """Recorre un Parquet remoto por lotes, descargando sólo lo que va leyendo.

    Es un generador: llamarlo no descarga nada, y el fichero se abre cuando alguien
    pide el primer lote. Así la descarga la hace quien lo recorre (el cargador, en
    su hilo, sin bloquear el cuaderno) y en memoria sólo está lo que se está
    leyendo, no el fichero entero: `Posts` ocupa más de 800 MB como tabla Arrow.
    """
    with fsspec.open(url, "rb") as parquet_file:
        yield from pq.ParquetFile(parquet_file).iter_batches(batch_size=batch_size)

Leemos los esquemas de los ficheros que vamos a cargar. Son peticiones muy pequeñas, así que terminan en un momento; los datos se descargarán después, por lotes, según se vayan insertando.

In [ ]:
posts_schema: pa.Schema = read_remote_parquet_schema(f"{PARQUET_BASE_URL}/Posts.parquet")
users_schema: pa.Schema = read_remote_parquet_schema(f"{PARQUET_BASE_URL}/Users.parquet")
tags_schema: pa.Schema = read_remote_parquet_schema(f"{PARQUET_BASE_URL}/Tags.parquet")

posts_schema

In [ ]:
# @title Diapositiva: Del esquema a las tablas
from IPython.display import HTML, display; display(HTML('<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#fff9db;color:#29261d;border:1px solid #dfcc78;border-top:10px solid #287681;border-radius:10px;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:22px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#287681" size="8" noshade style="display:none"><div style="color:#287681;font-size:15px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#287681" style="color:inherit"><b>Recapitulación</b></font></div><h1 style="color:#7b2c19;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:34px;line-height:1.15;margin:0 0 22px">Del esquema a las tablas</h1><ul style=\'margin:18px 0 8px;padding-left:30px\'><li style=\'margin:10px 0;list-style-position:outside\'>El <em>footer</em> de Parquet aporta nombres, tipos y nulabilidad</li><li style=\'margin:10px 0;list-style-position:outside\'>Esos metadatos no fijan por sí solos la clave de partición</li><li style=\'margin:10px 0;list-style-position:outside\'>Una misma entidad puede aparecer en varias tablas desnormalizadas</li><li style=\'margin:10px 0;list-style-position:outside\'>La consulta prevista será la prueba de cada diseño</li></ul><blockquote style=\'background:#fff0ae;color:inherit;border:none;border-left:6px solid #287681;margin:22px 0 0;padding:12px 16px\'><div style=\'margin-top:0px\'>Dejamos el formato de entrada y empezamos a modelar acceso</div></blockquote><hr color="#287681" size="2" noshade style="display:none"></div>'))

In [ ]:
# @title Diapositiva: Escribir para la consulta
from IPython.display import HTML, display; display(HTML('<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#fff9db;color:#29261d;border:1px solid #dfcc78;border-top:10px solid #c69200;border-radius:10px;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:22px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#c69200" size="8" noshade style="display:none"><div style="color:#c69200;font-size:15px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#c69200" style="color:inherit"><b>A continuación</b></font></div><h1 style="color:#7b2c19;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:34px;line-height:1.15;margin:0 0 22px">Escribir para la consulta</h1><ul style=\'margin:18px 0 8px;padding-left:30px\'><li style=\'margin:10px 0;list-style-position:outside\'>Abrir un <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>Session</code> y ejecutar CQL desde Python</li><li style=\'margin:10px 0;list-style-position:outside\'>Crear <em>keyspaces</em> y tablas con la clave primaria adecuada</li><li style=\'margin:10px 0;list-style-position:outside\'>Diferenciar clave de partición y columnas de <em>clustering</em></li><li style=\'margin:10px 0;list-style-position:outside\'>Preparar <em>statements</em> y agrupar escrituras concurrentes</li></ul><blockquote style=\'background:#fff0ae;color:inherit;border:none;border-left:6px solid #c69200;margin:22px 0 0;padding:12px 16px\'><div style=\'margin-top:0px\'>Una tabla Cassandra no es una tabla universal: sirve a un patrón de acceso</div></blockquote><hr color="#c69200" size="2" noshade style="display:none"></div>'))

## Uso básico de Cassandra desde Python

El *driver* de Cassandra para Python permite ejecutar consultas [CQL](https://cassandra.apache.org/doc/5.0/cassandra/developing/cql/index.html) directamente desde Python. La función principal es [`Session.execute()`](https://docs.datastax.com/en/developer/python-driver/latest/api/cassandra/cluster/#cassandra.cluster.Session.execute), que ejecuta una consulta CQL y devuelve los resultados.

Los resultados se devuelven en un objeto de tipo [`ResultSet`](https://docs.datastax.com/en/developer/python-driver/latest/api/cassandra/cluster/#cassandra.cluster.ResultSet), que se puede iterar para obtener las filas devueltas por la consulta. También tiene la opción de, si es un único resultado, usar `one()` para obtener la primera fila directamente. Y también se puede convertir a una lista con `all()`.

Cada una de las filas devueltas es un objeto similar a una tupla, donde se puede acceder a los valores de las columnas por su nombre o por su índice (`Row`).

In [ ]:
# @title Diapositiva: Conectar y ejecutar CQL
from IPython.display import HTML, display; display(HTML('<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#fff9db;color:#29261d;border:1px solid #dfcc78;border-top:10px solid #287681;border-radius:10px;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:22px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#287681" size="8" noshade style="display:none"><div style="color:#287681;font-size:15px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#287681" style="color:inherit"><b>Recapitulación</b></font></div><h1 style="color:#7b2c19;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:34px;line-height:1.15;margin:0 0 22px">Conectar y ejecutar CQL</h1><ul style=\'margin:18px 0 8px;padding-left:30px\'><li style=\'margin:10px 0;list-style-position:outside\'><code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>Session.execute</code> envía CQL y devuelve los resultados</li><li style=\'margin:10px 0;list-style-position:outside\'>Los <em>keyspaces</em> separan espacios de nombres y políticas de replicación</li><li style=\'margin:10px 0;list-style-position:outside\'>Las tablas se crean pensando en una consulta concreta</li><li style=\'margin:10px 0;list-style-position:outside\'>Los tipos y las claves deben coincidir con los valores que cargaremos</li></ul><blockquote style=\'background:#fff0ae;color:inherit;border:none;border-left:6px solid #287681;margin:22px 0 0;padding:12px 16px\'><div style=\'margin-top:0px\'>Ahora convertiremos esa decisión en un <em>keyspace</em> y varias tablas</div></blockquote><hr color="#287681" size="2" noshade style="display:none"></div>'))

In [ ]:
# @title Diapositiva: Crear el espacio y las tablas
from IPython.display import HTML, display; display(HTML('<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#fff9db;color:#29261d;border:1px solid #dfcc78;border-top:10px solid #c69200;border-radius:10px;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:22px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#c69200" size="8" noshade style="display:none"><div style="color:#c69200;font-size:15px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#c69200" style="color:inherit"><b>A continuación</b></font></div><h1 style="color:#7b2c19;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:34px;line-height:1.15;margin:0 0 22px">Crear el espacio y las tablas</h1><ul style=\'margin:18px 0 8px;padding-left:30px\'><li style=\'margin:10px 0;list-style-position:outside\'>Definir el <em>keyspace</em> con una estrategia de replicación adecuada</li><li style=\'margin:10px 0;list-style-position:outside\'>Crear primero las tablas que responden a consultas conocidas</li><li style=\'margin:10px 0;list-style-position:outside\'>Elegir partición y <em>clustering</em> antes de cargar datos</li><li style=\'margin:10px 0;list-style-position:outside\'>Comprobar tipos y nombres generados desde el esquema</li></ul><blockquote style=\'background:#fff0ae;color:inherit;border:none;border-left:6px solid #c69200;margin:22px 0 0;padding:12px 16px\'><div style=\'margin-top:0px\'>La clave primaria es la primera optimización de Cassandra</div></blockquote><hr color="#c69200" size="2" noshade style="display:none"></div>'))

## Creación de espacios de nombres y tablas

Vamos a crear ahora un `keyspace` llamado `stackoverflow` que vamos a usar posteriormente para definir las tablas necesarias. Un *keyspace* en Cassandra es similar a una base de datos en otros sistemas de gestión de bases de datos, ofrece un espacio de separación de nombres donde se pueden crear nuevas tablas. Los *keyspaces* también fijan las opciones de replicación, que se aplican a todas sus tablas: la celda siguiente elige la estrategia y el factor de replicación.

In [ ]:
from cassandra.cluster import ResultSet

# El factor de replicación es el número de nodos que ha descubierto el driver,
# con un máximo de 3: un nodo (Colab) tiene RF = 1 y los dos nodos del Compose,
# RF = 2. No se escribe a mano para que el mismo cuaderno valga en los dos casos.
replication_factor: int = min(nhosts, 3)

# SimpleStrategy basta para un solo datacenter y es la más sencilla. En
# producción, y siempre que haya varios datacenters, se usa NetworkTopologyStrategy,
# que fija un factor de replicación por datacenter y reparte las réplicas entre
# racks.
create_keyspace_cql: str = f"""
    CREATE KEYSPACE IF NOT EXISTS stackoverflow
    WITH replication = {{'class': 'SimpleStrategy', 'replication_factor': {replication_factor}}}
"""
print(create_keyspace_cql)

resultado: ResultSet = session.execute(create_keyspace_cql)
print(resultado.one())

Se utiliza ese *keyspace* a partir de ahora.

In [ ]:
session.set_keyspace("stackoverflow")

### Crear un `"CREATE TABLE"` con CQL para cada tabla

Construimos una función para generar los comandos CQL `CREATE TABLE` a partir de los esquemas que definimos anteriormente. Modificaremos esos esquemas para que se ajusten a las necesidades de Cassandra.

No hace falta leer la función línea a línea: traduce cada tipo de Arrow a su tipo CQL y compone el `CREATE TABLE`, que se imprime antes de ejecutarlo en cada tabla. Lo que importa es su parámetro `key_form`, que da la clave primaria: su primer elemento es la clave de partición (una columna, o una lista si es compuesta) y los siguientes, las columnas de *clustering* en orden.

El `CREATE TABLE` generado incluye `WITH compression = {'class': 'LZ4Compressor'}`. LZ4 ya es la compresión en disco por omisión de Cassandra, así que no cambia el resultado: la dejamos explícita porque la compresión se decide **por tabla**.

In [ ]:
from collections import OrderedDict

import pyarrow as pa


def cassandra_create_table_from_pyarrow(
    schema: pa.Schema,
    table_name: str,
    key_form: str | list[str | list[str]] | None = None,
    exclude_fields: list[str] | None = None,
    clustering_order: dict[str, str] | None = None,
    keyspace: str | None = None,
) -> str:
    """
    Generate a CREATE TABLE CQL statement from a PyArrow schema.

    Args:
        schema: pyarrow.Schema instance.
        table_name: target table name (no keyspace).
        key_form: list of lists of column names. First inner list is partition key columns,
                  remaining inner lists are flattened and used as clustering keys in order.
                  Example: [['user_id'], 'bucket', 'created_at']
        exclude_fields: list of field names to skip.
        clustering_order: optional dict mapping clustering column -> "ASC" or "DESC".
        keyspace: optional keyspace name to prefix table.

    Returns:
        cql_create_statement: str
    """

    def _map_arrow_type_to_cassandra(t: pa.DataType) -> str:
        # Handle dictionary (encoded) types
        if pa.types.is_dictionary(t):
            # Get the value type of the dictionary and map that
            value_type = t.value_type
            return _map_arrow_type_to_cassandra(value_type)

        # Handle map types with key-value structure
        if pa.types.is_map(t):
            # Map type: map<key_type, value_type>
            # Recursively map key and value types
            key_type: str = _map_arrow_type_to_cassandra(t.key_type)
            value_type: str = _map_arrow_type_to_cassandra(t.item_type)
            return f"map<{key_type}, {value_type}>"

        # Handle list types
        if pa.types.is_list(t) or pa.types.is_large_list(t):
            # List type: list<element_type>
            element_type: str = _map_arrow_type_to_cassandra(t.value_type)
            return f"list<{element_type}>"

        # Handle struct types
        if pa.types.is_struct(t):
            # Struct type: frozen<map<text, text>> as fallback
            return "frozen<map<text, text>>"

        # primitive types
        if pa.types.is_int8(t):
            return "tinyint"
        if pa.types.is_int16(t):
            return "smallint"
        if pa.types.is_int32(t):
            return "int"
        if pa.types.is_int64(t):
            return "bigint"
        if pa.types.is_uint8(t):
            return "smallint"
        if pa.types.is_uint16(t):
            return "int"
        if pa.types.is_uint32(t):
            return "bigint"
        if pa.types.is_uint64(t):
            return "bigint"
        if pa.types.is_float16(t) or pa.types.is_float32(t):
            return "float"
        if pa.types.is_float64(t):
            return "double"
        if pa.types.is_boolean(t):
            return "boolean"
        if pa.types.is_string(t) or pa.types.is_large_string(t):
            return "text"
        if pa.types.is_binary(t) or pa.types.is_large_binary(t) or pa.types.is_fixed_size_binary(t):
            return "blob"
        if pa.types.is_timestamp(t):
            return "timestamp"
        if pa.types.is_date32(t) or pa.types.is_date64(t):
            return "date"
        if pa.types.is_time32(t) or pa.types.is_time64(t):
            return "time"
        if pa.types.is_decimal(t):
            return "decimal"
        # fallback
        return "text"

    exclude: set[str] = set(exclude_fields or [])
    columns: OrderedDict[str, str] = OrderedDict()
    for fld in schema:
        if fld.name in exclude:
            continue
        cass_t: str = _map_arrow_type_to_cassandra(fld.type)
        columns[fld.name] = cass_t

    if not columns:
        raise ValueError("No columns left after applying exclude_fields.")

    # Build PRIMARY KEY
    if key_form:
        if not isinstance(key_form, (str, list)) or not all(
            isinstance(k, (str, list)) for k in key_form
        ):
            raise ValueError("Invalid key_form format.")

        match key_form:
            case str():  # "key"
                partition_cols: list[str] = [key_form]
                clustering_cols: list[str] = []
            case list():  # ["<key>", "<clustering>"] or [["<key1>", "<key2>", ...], "clustering"]
                match key_form[0]:
                    case list():
                        partition_cols = key_form[0]
                    case str():
                        partition_cols = [key_form[0]]
                clustering_cols: list[str] = key_form[1:]  # type: ignore
    else:
        field_names: list[str] = [f.name for f in schema if f.name not in exclude]
        partition_cols = [field_names[0]]
        clustering_cols = field_names[1:]

    for col in partition_cols + clustering_cols:
        if col not in columns:
            raise ValueError(f"Primary key column '{col}' not found in schema (or was excluded).")

    cols_cql_lines: list[str] = [f"  {name} {typ}" for name, typ in columns.items()]
    cols_cql_str: str = ",\n".join(cols_cql_lines)

    if len(partition_cols) == 1:
        pk_left: str = partition_cols[0]
    else:
        pk_left = "(" + ", ".join(partition_cols) + ")"
    pk_rest: str = ", ".join(clustering_cols) if clustering_cols else ""
    pk_clause: str = f"PRIMARY KEY ({pk_left}" + ((", " + pk_rest) if pk_rest else "") + ")"

    clustering_clause = ""
    if clustering_cols and clustering_order:
        orders: list[str] = []
        for c in clustering_cols:
            ordv: str | None = clustering_order.get(c)
            if ordv:
                ordv_upper: str = ordv.upper()
                if ordv_upper not in ("ASC", "DESC"):
                    raise ValueError("clustering_order values must be 'ASC' or 'DESC'")
                orders.append(f"{c} {ordv_upper}")
        if orders:
            clustering_clause: str = " \nAND CLUSTERING ORDER BY (" + ", ".join(orders) + ")"

    ks_prefix: str = f"{keyspace}." if keyspace else ""
    create: str = (
        f"CREATE TABLE IF NOT EXISTS {ks_prefix}{table_name} (\n{cols_cql_str},\n  {pk_clause}\n)"
        + " WITH compression = {'class': 'LZ4Compressor'} "
        + f"{clustering_clause};"
    )
    return create

### Generar los `INSERT INTO` para cada tabla

La función `cassandra_insert_template()` genera un `INSERT INTO` con marcadores de posición (`?`) para cada columna, y devuelve la plantilla y la lista de nombres de columnas. La plantilla se usa con `session.execute(prepared, values)` para insertar cada fila.

In [ ]:
def cassandra_insert_template(
    schema: pa.Schema,
    table_name: str,
    exclude_fields: set[str] | None = None,
    keyspace: str | None = None,
) -> tuple[str, list[str]]:
    """
    Construye una plantilla CQL de INSERT para Cassandra (placeholders posicionales '?') a partir
    de un esquema PyArrow.

    Devuelve (insert_cql, columns), donde:
      - insert_cql es, por ejemplo, "INSERT INTO ks.table (c1,c2) VALUES (?,?);"
      - columns son las columnas del INSERT, en el orden definido por el esquema: los
        valores deben enviarse en ese orden.

    Notas:
      - Los campos indicados en exclude_fields se omiten.
      - No realiza mapeo de tipos ni prepara la sentencia contra un clúster.
    """
    exclude: set[str] = exclude_fields or set()
    cols: list[str] = [field.name for field in schema if field.name not in exclude]

    if not cols:
        raise ValueError("No columns remain after applying exclude_fields.")

    ks_prefix: str = f"{keyspace}." if keyspace else ""
    col_list: str = ", ".join(cols)
    placeholders: str = ", ".join("?" for _ in cols)
    cql: str = f"INSERT INTO {ks_prefix}{table_name} ({col_list}) VALUES ({placeholders});"
    return cql, cols

### Tabla `Tags`

Vamos a crear la tabla `Tags`. La clave primaria es el `Id` de la etiqueta. La consulta se hace por `Id`, y también por `TagName`. Veremos cómo crear un índice adicional.

In [ ]:
tags_cql_create: str = cassandra_create_table_from_pyarrow(
    tags_schema,
    table_name="Tags",
    key_form=["Id"],
    exclude_fields=None,
    clustering_order=None,
    keyspace="stackoverflow",
)
print(tags_cql_create)

In [ ]:
session.execute(tags_cql_create)

In [ ]:
# @title Diapositiva: Identidad, partición y orden
from IPython.display import HTML, display; display(HTML('<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#fff9db;color:#29261d;border:1px solid #dfcc78;border-top:10px solid #287681;border-radius:10px;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:22px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#287681" size="8" noshade style="display:none"><div style="color:#287681;font-size:15px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#287681" style="color:inherit"><b>Recapitulación</b></font></div><h1 style="color:#7b2c19;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:34px;line-height:1.15;margin:0 0 22px">Identidad, partición y orden</h1><ul style=\'margin:18px 0 8px;padding-left:30px\'><li style=\'margin:10px 0;list-style-position:outside\'>El <em>keyspace</em> fija la estrategia de replicación</li><li style=\'margin:10px 0;list-style-position:outside\'>Cada tabla declara una clave de partición y, si hace falta, <em>clustering</em></li><li style=\'margin:10px 0;list-style-position:outside\'>Las consultas válidas deben proporcionar las restricciones que el modelo espera</li><li style=\'margin:10px 0;list-style-position:outside\'>Ahora veremos cómo reutilizar la misma forma con statements preparados</li></ul><blockquote style=\'background:#fff0ae;color:inherit;border:none;border-left:6px solid #287681;margin:22px 0 0;padding:12px 16px\'><div style=\'margin-top:0px\'>La clave no sólo identifica: también decide dónde y cómo se lee</div></blockquote><hr color="#287681" size="2" noshade style="display:none"></div>'))

In [ ]:
# @title Diapositiva: ¿Qué garantiza una consulta Cassandra?
from IPython.display import HTML, display; display(HTML('<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#fff9db;color:#29261d;border:1px solid #dfcc78;border-top:10px solid #a33a2b;border-radius:10px;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:22px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#a33a2b" size="8" noshade style="display:none"><div style="color:#a33a2b;font-size:15px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#a33a2b" style="color:inherit"><b>Pregunta guía</b></font></div><h1 style="color:#7b2c19;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:34px;line-height:1.15;margin:0 0 22px">¿Qué garantiza una consulta Cassandra?</h1><ul style=\'margin:18px 0 8px;padding-left:30px\'><li style=\'margin:10px 0;list-style-position:outside\'>La clave de partición decide dónde se busca</li><li style=\'margin:10px 0;list-style-position:outside\'>El <em>clustering</em> decide el orden dentro de la partición</li><li style=\'margin:10px 0;list-style-position:outside\'>Un <em>statement</em> preparado fija forma y tipos</li><li style=\'margin:10px 0;list-style-position:outside\'>El paralelismo debe respetar el tamaño de cada partición</li></ul><blockquote style=\'background:#fff0ae;color:inherit;border:none;border-left:6px solid #a33a2b;margin:22px 0 0;padding:12px 16px\'><div style=\'margin-top:0px\'>Antes de escribir el bucle, fijemos el patrón de lectura</div></blockquote><hr color="#a33a2b" size="2" noshade style="display:none"></div>'))

### Consultas, *statements* preparados

Las consultas en Cassandra se realizan mediante CQL (Cassandra Query Language), que es similar a SQL pero con algunas diferencias. Vamos a ver cómo realizar consultas básicas en Cassandra desde Python. Las consultas se pueden enviar directamente en texto, pero es recomendable utilizar *statements* preparados para mejorar el rendimiento y la seguridad. En Cassandra, los *statements* preparados se crean con `session.prepare()`, que devuelve un objeto `PreparedStatement`. Luego, para ejecutar el *statement* preparado, se utiliza `session.execute(prepared_statement, parameters)`, donde `parameters` es una tupla con los valores a sustituir en la consulta. El uso de *statements* preparados permite a Cassandra optimizar la ejecución de las consultas, evitar inyecciones de código y, además, es lo que permite al `TokenAwarePolicy` saber a qué réplica enviar cada escritura.

El *driver* también permite enviar muchas consultas (en este caso, inserciones) sin esperar a que termine cada una: `session.execute_async()` envía la consulta y devuelve enseguida un `ResponseFuture`, cuyo método `result()` espera la respuesta y lanza la excepción si ha fallado.

#### Escribir muchas filas: `populate_table()`

Poblar una tabla de Cassandra es casi siempre lo mismo: se parte de unas filas de origen (un lote leído del Parquet, una lista construida en Python o el resultado de un `SELECT` sobre otra tabla), se convierte cada una en los valores de un `INSERT` (o de un `UPDATE`) preparado y se envían todas de forma concurrente. Para no repetir ese código en cada tabla lo recogemos en una función, `populate_table()`, que usaremos en el cargador de los Parquet, en las tablas derivadas y en las soluciones de los ejercicios:

```python
filas_escritas = populate_table(session, statement, rows, execution_profile=..., to_values=None, concurrency=100)
```

- `statement`: el `INSERT` o `UPDATE` preparado con `session.prepare()`. Cada fila aporta los valores de sus `?`, en el mismo orden.
- `rows`: las filas de origen, en cualquier iterable: una **lista** de tuplas, un **generador** o el **`ResultSet`** de un `SELECT`. Se recorre una sola vez, sin guardarlo. Un `ResultSet` va página a página: en memoria sólo hay una página (`fetch_size` filas, 5000 por omisión), nunca la tabla entera.
- `execution_profile`: el perfil con el que se escribe. `LOAD_PROFILE` (consistencia `ANY` y *timeout* largo) para una carga masiva, o `EXEC_PROFILE_DEFAULT` (`LOCAL_QUORUM`) cuando lo escrito se va a leer enseguida, porque `ANY` no garantiza esa lectura. No tiene valor por omisión, para que la decisión quede a la vista en cada llamada.
- `to_values` (opcional): una función que recibe una fila de origen y devuelve la tupla de valores del `statement`, o `None` para saltarse esa fila. Sin ella, cada fila se envía tal cual, y sus columnas tienen que estar ya en el orden de los `?`.
- `concurrency` (opcional): cuántas escrituras puede haber en vuelo a la vez.

Devuelve el número de filas escritas.

Por dentro, la función recorre `rows` en el hilo principal y convierte cada fila justo antes de enviarla con `execute_async()`. Guarda en una cola (`deque`) las escrituras en vuelo: cuando hay `concurrency`, espera con `result()` a la más antigua antes de enviar otra, así que las filas no se acumulan en memoria. Al terminar, espera a las que quedan. Una escritura fallida lanza su excepción en `result()`, y un error en `to_values` se propaga como en cualquier bucle.

El *driver* ofrece también `execute_concurrent_with_args()`, que hace este reparto por nosotros, pero pide cada fila siguiente desde su hilo de red. Con una lista no hay problema, aunque obliga a construirla antes. Con un generador sí lo hay: si para dar la fila siguiente hay que traer una página o hacer una consulta (como `get_user()`, más adelante), ese hilo se queda esperando una respuesta que él mismo tendría que recibir, y la celda se cuelga. Y si `to_values` lanza una excepción, ésta se pierde: la llamada termina sin error y con menos filas de las debidas.

In [ ]:
from collections import deque
from collections.abc import Callable, Iterable, Sequence
from typing import Any

from cassandra.cluster import ResponseFuture, Session
from cassandra.query import PreparedStatement


def populate_table(
    session: Session,
    statement: PreparedStatement,
    rows: Iterable[Any],
    *,
    execution_profile: str,
    to_values: Callable[[Any], Sequence[Any] | None] | None = None,
    concurrency: int = 100,
) -> int:
    """Escribe con `statement`, de forma concurrente, una fila por cada fila de `rows`.

    `rows` es cualquier iterable (una lista, un generador, el `ResultSet` de un
    `SELECT`), que se recorre una sola vez, sin guardarlo. `to_values` convierte
    cada fila de origen en los valores de los `?` de `statement`, o devuelve
    `None` para saltarla; sin ella, se envía tal cual.
    Devuelve el número de filas escritas.
    """
    in_flight: deque[ResponseFuture] = deque()
    written: int = 0
    for row in rows:
        row_values: Sequence[Any] | None = row if to_values is None else to_values(row)
        if row_values is None:
            continue
        # Con el cupo lleno, se espera a la escritura más antigua antes de enviar
        # otra. result() lanza la excepción de la escritura si ha fallado.
        if len(in_flight) == concurrency:
            in_flight.popleft().result()
        in_flight.append(
            session.execute_async(statement, row_values, execution_profile=execution_profile)
        )
        written += 1
    for future in in_flight:
        future.result()
    return written

In [ ]:
# @title Diapositiva: Poblar una tabla con populate_table()
from IPython.display import HTML, display; display(HTML('<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#fff9db;color:#29261d;border:1px solid #dfcc78;border-top:10px solid #287681;border-radius:10px;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:22px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#287681" size="8" noshade style="display:none"><div style="color:#287681;font-size:15px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#287681" style="color:inherit"><b>Recapitulación</b></font></div><h1 style="color:#7b2c19;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:34px;line-height:1.15;margin:0 0 22px">Poblar una tabla con <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>populate_table()</code></h1><p style=\'margin:10px 0\'><code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>populate_table(session, statement, rows, execution_profile=..., to_values=None, concurrency=100)</code></p><ul style=\'margin:18px 0 8px;padding-left:30px\'><li style=\'margin:10px 0;list-style-position:outside\'><code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>statement</code>: <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>INSERT</code> o <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>UPDATE</code> preparado; cada fila aporta los valores de sus <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>?</code></li><li style=\'margin:10px 0;list-style-position:outside\'><code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>rows</code>: cualquier iterable (lista, generador o <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>ResultSet</code> de un <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>SELECT</code>), que se recorre sin guardarlo</li><li style=\'margin:10px 0;list-style-position:outside\'><code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>execution_profile</code> (obligatorio): <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>LOAD_PROFILE</code> para una carga masiva, <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>EXEC_PROFILE_DEFAULT</code> si se va a leer enseguida</li><li style=\'margin:10px 0;list-style-position:outside\'><code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>to_values</code>: convierte cada fila de origen en sus valores, o devuelve <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>None</code> para saltarla</li><li style=\'margin:10px 0;list-style-position:outside\'><code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>concurrency</code>: cuántas escrituras puede haber en vuelo a la vez</li><li style=\'margin:10px 0;list-style-position:outside\'>Devuelve el número de filas escritas</li></ul><blockquote style=\'background:#fff0ae;color:inherit;border:none;border-left:6px solid #287681;margin:22px 0 0;padding:12px 16px\'><div style=\'margin-top:0px\'>Por dentro, <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>execute_async()</code> y una cola con, como mucho, <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>concurrency</code> escrituras en vuelo</div></blockquote><hr color="#287681" size="2" noshade style="display:none"></div>'))

### De Parquet a Cassandra: `background_import_parquet_batches()`

La carga tiene dos piezas separadas. `iter_remote_parquet_batches()` recorre el Parquet remoto por lotes (`RecordBatch`) y descarga sólo lo que va leyendo: es un iterador, así que crearlo no descarga nada. Cada lote se puede transformar por el camino con operaciones de PyArrow (lo haremos con `Posts`, para recortar el campo `Body`). El cargador siguiente recibe ese iterador, lo recorre **una sola vez** y, por cada `RecordBatch`:

1. filtra las filas que corresponden a cada tabla destino (con `pyarrow.compute`, de forma vectorizada: `Posts.parquet` alimenta a la vez `Questions` y `Answers`);
2. selecciona las columnas en el mismo orden que el `INSERT` preparado;
3. convierte cada columna con `to_pylist()` y transpone con `zip()` para obtener las tuplas que espera el *driver*. Los nulos de Arrow —incluidas las fechas— se convierten directamente a `None`, así que aquí nunca aparecen `NaT` ni `NaN`, y se sustituyen por `UNSET_VALUE` para no escribirlos: en Cassandra un nulo explícito se guarda como una *tombstone*, igual que un borrado. Sin esa sustitución, cada pregunta dejaría unas cinco, y un recorrido de `Questions` saltaría el aviso `tombstone_warn_threshold` en cada página;
4. envía el lote con `populate_table()` usando el perfil de carga (consistencia `ANY`).

Esa conversión a listas de Python sí materializa objetos en memoria, pero sólo los de un lote: el coste está acotado por `batch_size`, no por el tamaño del fichero.

In [ ]:
from asyncio import Task, create_task, to_thread
from collections.abc import Iterable
from dataclasses import dataclass
from queue import Queue

import pyarrow.compute as pc
from cassandra.query import UNSET_VALUE, PreparedStatement


@dataclass
class LoadTarget:
    """Tabla destino de una carga: sentencia preparada, columnas y filtro de filas."""

    name: str
    statement: PreparedStatement
    columns: list[str]
    # Par (columna, valor): sólo se cargan las filas cuya columna valga eso.
    row_filter: tuple[str, int] | None = None


def background_import_parquet_batches(
    session: Session,
    batches: Iterable[pa.RecordBatch],
    targets: list[LoadTarget],
    logqueue: Queue[str] | None = None,
    concurrency: int = 100,
    execution_profile: str = LOAD_PROFILE,
) -> Task[dict[str, int]]:
    """Carga en Cassandra los lotes de un Parquet, en un hilo y sin bloquear el cuaderno.

    Recorre `batches` una sola vez y reparte cada lote entre sus tablas destino.
    Recibe los lotes ya preparados y no sabe de qué fichero vienen ni si se han
    transformado por el camino: leer, transformar y cargar quedan separados, y lo
    particular de un fichero (recortar una columna, por ejemplo) se decide en la
    celda que lanza la carga, no aquí. Como `batches` se recorre dentro del hilo,
    si es un iterador perezoso la descarga también ocurre en segundo plano.
    Devuelve una `Task` que da, al terminar, las filas insertadas por tabla.
    """

    def _column_to_bind_values(column: pa.Array) -> list[object]:
        """Convierte una columna Arrow en valores para el INSERT, sin escribir los nulos.

        Enlazar `None` escribe un nulo explícito, y Cassandra lo guarda como una
        *tombstone* de celda: la misma marca que deja un borrado, que cada lectura
        tiene que saltarse hasta que la compactación la elimina. Con `UNSET_VALUE`
        el *driver* no envía esa columna: la celda no se escribe y al leerla sale
        `None` igualmente. Sólo vale para una carga inicial como ésta: para vaciar
        una columna que ya tiene valor sí hay que escribir el nulo.

        La sustitución se hace por columnas y no por filas: `null_count` lo calcula
        Arrow sobre su mapa de validez, sin pasar por Python, así que las columnas
        sin nulos (la clave, entre otras) se devuelven tal cual.
        """
        values: list[object] = column.to_pylist()
        if column.null_count == 0:
            return values
        return [UNSET_VALUE if value is None else value for value in values]

    def _log(message: str) -> None:
        if logqueue is not None:
            logqueue.put_nowait(message)

    def _load() -> dict[str, int]:
        inserted: dict[str, int] = {target.name: 0 for target in targets}
        try:
            for record_batch in batches:
                for target in targets:
                    batch: pa.RecordBatch = record_batch
                    if target.row_filter:
                        column, value = target.row_filter
                        batch = batch.filter(pc.equal(batch.column(column), value))
                    if batch.num_rows == 0:
                        continue
                    # select() garantiza el mismo orden que el INSERT preparado.
                    batch = batch.select(target.columns)
                    # Arrow convierte los nulos (también las fechas) a None; a
                    # diferencia de pandas, aquí nunca aparecen NaT ni NaN.
                    values: list[list[object]] = [
                        _column_to_bind_values(column) for column in batch.columns
                    ]
                    _log(f"{target.name}: enviando {batch.num_rows:,} filas")
                    populate_table(
                        session,
                        target.statement,
                        zip(*values),
                        execution_profile=execution_profile,
                        concurrency=concurrency,
                    )
                    inserted[target.name] += batch.num_rows
                    total: int = inserted[target.name]
                    _log(f"{target.name}: filas insertadas: {total:,}")
        except Exception as exc:
            _log(f"Error durante la carga: {exc!r}")
            raise
        return inserted

    return create_task(to_thread(_load))


def print_log(logqueue: Queue[str]) -> None:
    """Vacía la cola de progreso de una carga."""
    while not logqueue.empty():
        print(logqueue.get_nowait())

Se hace la inserción de los datos en la tabla `Tags`. Se podría hacer con `session.execute()` en un bucle, pero aquí se muestra cómo hacerlo con una llamada que lo importa en segundo plano, aunque después esperemos a que termine. Para las tablas más grandes sí que será interesante que se haga en segundo plano.

In [ ]:
tags_insert_cql: str
tags_columns: list[str]
tags_insert_cql, tags_columns = cassandra_insert_template(tags_schema, "stackoverflow.Tags")

tags_target: LoadTarget = LoadTarget(
    name="Tags",
    statement=session.prepare(tags_insert_cql),
    columns=tags_columns,
)

tags_log_queue: Queue[str] = Queue()
future_tags: Task[dict[str, int]] = background_import_parquet_batches(
    session,
    iter_remote_parquet_batches(f"{PARQUET_BASE_URL}/Tags.parquet"),
    [tags_target],
    logqueue=tags_log_queue,
)

In [ ]:
import asyncio

try:
    tags_inserted: dict[str, int] = await asyncio.wait_for(future_tags, timeout=300)
except TimeoutError:
    print("Timeout waiting for tags import to complete.")
    raise

print_log(tags_log_queue)
print(f"Tags importados: {tags_inserted}")

Y veamos una consulta de selección sencilla.

In [ ]:
tags_result: ResultSet = session.execute("""
    SELECT Id, TagName, Count
    FROM stackoverflow.Tags
    WHERE Id IN (1,2,3,4,5,6,7,8,9,10)
""")
for row in tags_result:
    print(row.id, row.tagname, row.count)

### Tablas `Questions` y `Answers`

A continuación vamos a crear las tablas `Questions` y `Answers`. Las dos salen del mismo fichero, `Posts.parquet`, donde preguntas y respuestas conviven y se distinguen por `PostTypeId`. ¿Por qué no crear una sola tabla `Posts`, como haríamos en SQL?

En Cassandra las tablas no se diseñan a partir de las entidades, sino de las consultas que deben servir (modelado dirigido por consultas). Una tabla sólo responde de forma eficiente a las consultas que indican su clave de partición, y no hay `JOIN` para combinar tablas al consultar. Aquí nos interesan dos consultas distintas:

- recuperar una pregunta a partir de su `Id`;
- recuperar todas las respuestas de una pregunta.

La primera pide una tabla particionada por el `Id` de la pregunta; la segunda, una tabla particionada por `ParentId`, de modo que todas las respuestas de una pregunta queden juntas en la misma partición y se lean con una sola consulta. Con una única tabla `Posts` particionada por `Id`, obtener las respuestas de una pregunta obligaría a recorrer todas las particiones.

In [ ]:
questions_create_cql: str = cassandra_create_table_from_pyarrow(
    posts_schema,
    table_name="Questions",
    key_form=["Id", "CreationDate"],
    exclude_fields=["PostTypeId", "ParentId"],
    clustering_order={"CreationDate": "ASC"},
    keyspace="stackoverflow",
)
print(questions_create_cql)

La clave primaria de `Questions` es `(Id, CreationDate)`: `Id` es la clave de partición y `CreationDate`, una columna de *clustering* con orden ascendente. Como `Id` es único, cada partición contiene una sola pregunta, así que `CreationDate` no aporta nada a la identidad (el `Id` ya basta) ni al orden (no hay otras filas en la partición que ordenar). Sí tiene un coste: una columna de la clave no se puede modificar, y para actualizar o borrar una pregunta concreta hay que dar las dos columnas (`WHERE Id = ? AND CreationDate = ?`), no sólo el `Id`.

¿Por qué está entonces? Por las *materialized views*, que se explican en la teoría. La clave de una vista debe contener todas las columnas de la clave de la tabla base y, como mucho, una columna más. Con `CreationDate` en la clave de `Questions`, una vista puede particionar por autor (`OwnerUserId`, la columna añadida) y ordenar sus preguntas por fecha; si la clave base fuera sólo `Id`, la vista no podría usar a la vez el autor y la fecha. Aquí no creamos ninguna vista, pero la clave conserva esa forma. Es algo habitual en Cassandra: la clave de una tabla no se elige sólo por las consultas que sirve ella, sino también por las tablas y vistas que se derivan de ella.

Con `session.execute()` ejecutamos los comandos CQL para crear la tabla.

In [ ]:
session.execute(questions_create_cql)

In [ ]:
questions_excluded_fields: set[str] = {
    "PostTypeId",
    "ParentId",
}

# PostTypeId == 1 son las preguntas. La columna se usa para filtrar aunque no
# se inserte: el cargador la pide al Parquet y la descarta al hacer select().
questions_insert_cql: str
questions_columns: list[str]
questions_insert_cql, questions_columns = cassandra_insert_template(
    posts_schema, "stackoverflow.Questions", exclude_fields=questions_excluded_fields
)

questions_target: LoadTarget = LoadTarget(
    name="Questions",
    statement=session.prepare(questions_insert_cql),
    columns=questions_columns,
    row_filter=("PostTypeId", 1),
)

print(questions_insert_cql)

Vamos ahora a crear la tabla de respuestas (`Answers`). La clave de partición es `ParentId` (la pregunta a la que responde), porque las respuestas se consultan siempre a partir de su pregunta: así todas las respuestas de una pregunta quedan juntas en la misma partición y se recuperan con una sola consulta. Dentro de la partición, la columna de *clustering* `CreationDate` las ordena de la más antigua a la más reciente.

La clave primaria completa es `(ParentId, CreationDate)`. Supone que dos respuestas a la misma pregunta no se publican en el mismo milisegundo, algo que se cumple en este *dump*. Si no fuera así, la segunda escritura sustituiría en silencio a la primera: en Cassandra un `INSERT` con una clave que ya existe es una actualización, no un error. Añadir `Id` al final de la clave lo evitaría.

In [ ]:
answers_cql_create: str = cassandra_create_table_from_pyarrow(
    posts_schema,
    table_name="Answers",
    key_form=["ParentId", "CreationDate"],
    exclude_fields=["AcceptedAnswerId", "AnswerCount", "ClosedDate", "PostTypeId", "Tags", "Title"],
    clustering_order={"CreationDate": "ASC"},
    keyspace="stackoverflow",
)
print(answers_cql_create)

In [ ]:
session.execute(answers_cql_create)

Con las dos tablas creadas, cargamos ambas **en un solo recorrido** de `Posts.parquet`: cada lote se filtra por `PostTypeId` y se inserta en la tabla que corresponde. Este patrón —escribir la misma entidad en varias tablas modeladas según la consulta que deben servir— es exactamente el que se sigue en Cassandra al alimentar tablas desnormalizadas, y evita descargar dos veces el fichero.

In [ ]:
answers_excluded_fields: set[str] = {
    "AcceptedAnswerId",
    "AnswerCount",
    "ClosedDate",
    "PostTypeId",
    "Tags",
    "Title",
}

# PostTypeId == 2 son las respuestas.
answers_insert_cql: str
answers_columns: list[str]
answers_insert_cql, answers_columns = cassandra_insert_template(
    posts_schema, "stackoverflow.Answers", exclude_fields=answers_excluded_fields
)

answers_target: LoadTarget = LoadTarget(
    name="Answers",
    statement=session.prepare(answers_insert_cql),
    columns=answers_columns,
    row_filter=("PostTypeId", 2),
)

# El cuerpo de las publicaciones se guarda recortado a MAX_CARACTERES_BODY
# caracteres.
MAX_CARACTERES_BODY: int = 80

posts_log_queue: Queue[str] = Queue()
future_posts: Task[dict[str, int]] = background_import_parquet_batches(
    session,
    # El recorte se aplica lote a lote con map(), que es perezoso: aquí no se
    # descarga ni se transforma nada, sino en el hilo de la carga, según se lee
    # cada lote. Es una operación de Arrow sobre la columna entera, sin pasar por
    # objetos de Python. Va escrito dentro de la llamada, y no en una variable,
    # porque un iterador sólo se puede recorrer una vez: así cada ejecución de
    # la carga crea el suyo y no puede reutilizarse uno ya agotado.
    map(
        lambda batch: batch.set_column(
            batch.schema.get_field_index("Body"),
            "Body",
            pc.utf8_slice_codeunits(batch.column("Body"), start=0, stop=MAX_CARACTERES_BODY),
        ),
        iter_remote_parquet_batches(f"{PARQUET_BASE_URL}/Posts.parquet"),
    ),
    [questions_target, answers_target],
    logqueue=posts_log_queue,
)

In [ ]:
# ¿Terminó? Mientras tanto se puede ir viendo el progreso de la carga.
print(f"Carga de Posts terminada: {future_posts.done()}")
print_log(posts_log_queue)

### Tabla `Users`

Vamos a crear la tabla `Users`. En este caso, la clave primaria es el `Id` del usuario. La consulta se hace por `Id`, por lo que no es necesario crear un índice adicional.


In [ ]:
users_cql_create: str = cassandra_create_table_from_pyarrow(
    users_schema,
    table_name="Users",
    key_form=["Id"],
    exclude_fields=None,
    clustering_order=None,
    keyspace="stackoverflow",
)
print(users_cql_create)

In [ ]:
session.execute(users_cql_create)

La carga de `Users` no empieza ahora, sino cuando termine la de `Posts`: la celda siguiente crea una tarea que espera a esa carga y lanza después la suya. Con pocos núcleos, como en Google Colab, el cuaderno y Cassandra se reparten la CPU, y dos cargas a la vez no terminan antes que una tras otra: en una prueba con dos hilos de CPU, el total fue prácticamente el mismo, pero `Posts` sola terminó en poco más de la mitad de tiempo. Así `Questions` y `Answers` están listas antes, y `Users` se carga mientras trabajamos con ellas.

In [ ]:
import asyncio

users_insert_cql: str
users_columns: list[str]
users_insert_cql, users_columns = cassandra_insert_template(users_schema, "stackoverflow.Users")

users_target: LoadTarget = LoadTarget(
    name="Users",
    statement=session.prepare(users_insert_cql),
    columns=users_columns,
)

users_log_queue: Queue[str] = Queue()


async def import_users_after_posts() -> dict[str, int]:
    """Espera a que termine la carga de Posts y lanza entonces la de Users."""
    users_log_queue.put_nowait("Users: esperando a que termine la carga de Posts")
    # asyncio.wait() no propaga el error de Posts: ése lo muestra su propia espera.
    await asyncio.wait((future_posts,))
    return await background_import_parquet_batches(
        session,
        iter_remote_parquet_batches(f"{PARQUET_BASE_URL}/Users.parquet"),
        [users_target],
        logqueue=users_log_queue,
    )


future_users: Task[dict[str, int]] = create_task(import_users_after_posts())

In [ ]:
print(f"Carga de Users terminada: {future_users.done()}")
print_log(users_log_queue)

In [ ]:
# @title Diapositiva: Consultas y carga por lotes
from IPython.display import HTML, display; display(HTML('<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#fff9db;color:#29261d;border:1px solid #dfcc78;border-top:10px solid #287681;border-radius:10px;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:22px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#287681" size="8" noshade style="display:none"><div style="color:#287681;font-size:15px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#287681" style="color:inherit"><b>Recapitulación</b></font></div><h1 style="color:#7b2c19;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:34px;line-height:1.15;margin:0 0 22px">Consultas y carga por lotes</h1><ul style=\'margin:18px 0 8px;padding-left:30px\'><li style=\'margin:10px 0;list-style-position:outside\'>Las tablas separadas <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>Questions</code> y <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>Answers</code> responden a consultas distintas</li><li style=\'margin:10px 0;list-style-position:outside\'>El <em>statement</em> preparado evita reconstruir CQL por cada fila</li><li style=\'margin:10px 0;list-style-position:outside\'><code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>populate_table()</code> acota y solapa las peticiones</li><li style=\'margin:10px 0;list-style-position:outside\'>El mismo Parquet puede alimentar varias tablas durante un único recorrido</li></ul><blockquote style=\'background:#fff0ae;color:inherit;border:none;border-left:6px solid #287681;margin:22px 0 0;padding:12px 16px\'><div style=\'margin-top:0px\'>Hemos optimizado el camino del driver; ahora veremos qué puede aportar un índice</div></blockquote><hr color="#287681" size="2" noshade style="display:none"></div>'))

In [ ]:
# @title Diapositiva: Índices SAI
from IPython.display import HTML, display; display(HTML('<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#fff9db;color:#29261d;border:1px solid #dfcc78;border-top:10px solid #c69200;border-radius:10px;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:22px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#c69200" size="8" noshade style="display:none"><div style="color:#c69200;font-size:15px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#c69200" style="color:inherit"><b>A continuación</b></font></div><h1 style="color:#7b2c19;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:34px;line-height:1.15;margin:0 0 22px">Índices SAI</h1><ul style=\'margin:18px 0 8px;padding-left:30px\'><li style=\'margin:10px 0;list-style-position:outside\'>Crear índices sobre columnas que no forman la clave primaria, como en SQL</li><li style=\'margin:10px 0;list-style-position:outside\'>Esperar a que el índice se construya, no sólo a que se propague el esquema</li><li style=\'margin:10px 0;list-style-position:outside\'>Buscar por igualdad y por rango</li><li style=\'margin:10px 0;list-style-position:outside\'>Comprobar cuándo una consulta necesita <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>ALLOW FILTERING</code></li><li style=\'margin:10px 0;list-style-position:outside\'>Comparar el índice con una tabla diseñada para la misma pregunta</li></ul><blockquote style=\'background:#fff0ae;color:inherit;border:none;border-left:6px solid #c69200;margin:22px 0 0;padding:12px 16px\'><div style=\'margin-top:0px\'>Un índice no convierte Cassandra en una base de datos relacional</div></blockquote><hr color="#c69200" size="2" noshade style="display:none"></div>'))

### Índices SAI

La carga de `Posts` sigue en segundo plano y, cuando termine, empezará la de `Users`: entre las dos tardan varios minutos (en Google Colab, unos nueve). No hace falta esperarlas para seguir: `Tags` ya está cargada, así que empezamos los índices por ella, y la espera llega cuando una celda necesita las tablas grandes. Mientras tanto, puedes volver a ejecutar las celdas de progreso de más arriba.

Como en una base de datos SQL, un índice permite buscar por una columna que no es de la clave sin recorrer la tabla, y se crea con `CREATE INDEX`. Los índices actuales de Cassandra son los **SAI** (*Storage-Attached Indexing*, desde la 5.0), que se piden con `USING 'sai'`.

Un índice SAI admite:

- **Igualdad**: `WHERE TagName = '.net'`
- **Rangos en columnas numéricas**: `>`, `>=`, `<`, `<=`
- **Colecciones**: `CONTAINS` y `CONTAINS KEY`

No admite `LIKE` ni ordena el resultado por la columna indexada. Cassandra también permite combinar varias columnas indexadas con `AND`, pero aquí no lo usamos: en la 5.0.9, con un nivel de consistencia superior a `ONE`, esas consultas agotan el plazo (`ReadTimeout`). Como en SQL, cada índice añade coste a las escrituras (y aquí, también a la compactación). No se puede crear sobre una clave de partición de una sola columna, que ya se consulta directamente, ni en una tabla con columnas `counter`.

La gran diferencia con SQL es que **el índice es local**: se guarda junto a los datos de cada nodo (junto a cada *SSTable* y a la *memtable*) y sólo cubre lo que guarda ese nodo. Una consulta que no da la clave de partición tiene que preguntar a muchos nodos, así que su coste crece con el tamaño del clúster.

In [ ]:
# Un cambio de esquema (CREATE, ALTER, DROP) no usa nivel de consistencia: el
# nodo que lo recibe lo propaga a todo el clúster y el *driver* espera (con un
# límite) a que los nodos coincidan en la versión del esquema. Por eso no se le
# pasa `consistency_level`, a diferencia de una lectura o una escritura.
# Igualdad sobre un texto: buscar una etiqueta por su nombre.
session.execute("""
    CREATE INDEX IF NOT EXISTS tags_tagname_sai
    ON stackoverflow.Tags (TagName)
    USING 'sai'
""")

In [ ]:
# Sobre una columna numérica, el índice sirve también para rangos.
session.execute("""
    CREATE INDEX IF NOT EXISTS tags_count_sai
    ON stackoverflow.Tags (Count)
    USING 'sai'
""")

`CREATE INDEX` confirma el cambio de esquema, pero el índice de los datos existentes se construye después. Una consulta prematura puede fallar con `IndexNotAvailableException`; una espera fija no demuestra que haya terminado.

`wait_for_schema_build()` vigila las tareas de `system_views.sstable_tasks` y comprueba con una consulta real que el índice responde. En un clúster, hace esa comprobación con **cada nodo como coordinador**: cada uno puede conocer en distinto momento el estado que anuncian los demás. Además, el *build* de un índice puede abortarse (por ejemplo, si una compactación retira una SSTable mientras se recorre) y entonces el índice no se construye nunca: con `recreate_indexes=True`, la función lo detecta (ningún *build* en marcha y la consulta de prueba sigue fallando) y vuelve a crear el índice. El código siguiente resuelve todo esto; para usar un índice sólo necesitamos llamar a la función.


In [ ]:
import time

from cassandra.metadata import IndexMetadata
from cassandra.protocol import ServerError

from cassandra import OperationTimedOut, RequestExecutionException


def wait_for_schema_build(
    session: Session,
    keyspace: str,
    probes: dict[str, str],
    kind_markers: tuple[str, ...] = ("index", "view"),
    timeout: float = 180.0,
    poll_interval: float = 2.0,
    recreate_indexes: bool = False,
    stall_timeout: float = 30.0,
) -> None:
    """Espera a que un índice o una vista materializada recién creados estén
    listos para consultarse.

    Un `CREATE INDEX`/`CREATE MATERIALIZED VIEW` sólo confirma que el cambio
    de esquema se propagó por
    el clúster; el *build* -recorrer las SSTables existentes para poblar el
    índice o la vista- es un proceso aparte en segundo plano. `probes` es un
    diccionario {nombre: consulta CQL de prueba}: cada consulta debe ser
    de bajo coste y no importa que no devuelva filas, sólo que no lance una
    excepción por índice o vista no disponibles.

    El *build* de un índice SAI puede abortarse (por ejemplo, si una
    compactación retira una SSTable mientras se recorre) y entonces el índice
    no se construye nunca. Con `recreate_indexes=True`, los nombres de `probes`
    son los de índices: si durante `stall_timeout` segundos no hay ningún
    *build* en marcha y la prueba sigue fallando, se da por fallido y se borra
    y vuelve a crear ese índice, una sola vez.
    """

    def _up_hosts(session: Session) -> list[Host]:
        """Nodos activos del clúster: cada uno hará de coordinador en los sondeos."""
        return [host for host in session.cluster.metadata.all_hosts() if host.is_up]

    def _probe_ok(session: Session, probe_cql: str) -> bool:
        """True si la consulta de prueba ya funciona con cada nodo como coordinador.

        Mientras un índice o una vista se construyen, la consulta de prueba puede
        fallar de varias formas según la versión y el nodo: `ReadFailure` (lo
        habitual para `IndexNotAvailableException`), `ReadTimeout`, `Unavailable`,
        un `ServerError` genérico o un *timeout* del propio *driver*. Se capturan
        por sus clases base: `RequestExecutionException` agrupa todos los errores
        de ejecución en el servidor. No se captura `InvalidRequest`: una consulta
        de prueba mal escrita debe fallar en el acto, no esperar hasta el plazo.

        La prueba se lanza una vez por nodo, fijando el coordinador con `host=`
        (que deja de lado el reparto de carga del perfil). Cada nodo publica por
        *gossip* el estado de sus índices, y el coordinador de una lectura descarta
        de antemano las réplicas cuyo índice cree todavía en construcción. Hasta
        que le llega la noticia (el *gossip* corre una vez por segundo), un nodo
        puede seguir rechazando con INDEX_NOT_AVAILABLE la consulta que otro ya
        coordina sin problemas: con una sola prueba, que el reparto de carga envía
        a un único nodo, los demás coordinadores quedarían sin comprobar.
        """
        try:
            for host in _up_hosts(session):
                session.execute(probe_cql, host=host)
            return True
        except (RequestExecutionException, ServerError, OperationTimedOut):
            return False

    def _background_build_running(
        session: Session, keyspace: str, kind_markers: tuple[str, ...]
    ) -> bool | None:
        """
        True/False según `system_views.sstable_tasks`, o None si esa tabla
        virtual no está disponible (versión de Cassandra distinta, permisos...).
        `kind_markers` filtra por subcadenas del campo `kind` (por ejemplo
        'index' para índices SAI, 'view' para vistas
        materializadas). Una tabla virtual es local: sólo describe las tareas
        del nodo que responde, así que se pregunta a todos, uno por uno.
        """
        try:
            kinds: list[str | None] = [
                row.kind
                for host in _up_hosts(session)
                for row in session.execute(
                    "SELECT kind FROM system_views.sstable_tasks WHERE keyspace_name = %s",
                    (keyspace,),
                    host=host,
                )
            ]
        except Exception:
            return None
        return any(
            kind and any(marker in kind.lower() for marker in kind_markers) for kind in kinds
        )

    def _recreate_index(session: Session, keyspace: str, index_name: str) -> None:
        """Borra un índice y lo vuelve a crear con la definición que guardan los
        metadatos del *driver*. Un fallo (por ejemplo, un índice que no existe) se propaga."""
        for table in session.cluster.metadata.keyspaces[keyspace].tables.values():
            index: IndexMetadata | None = table.indexes.get(index_name)
            if index is not None:
                # La definición se lee antes del DROP, que la quitaría de los metadatos.
                create_cql: str = index.as_cql_query()
                session.execute(f"DROP INDEX IF EXISTS {keyspace}.{index_name}")
                session.execute(create_cql)
                return
        raise ValueError(f"El índice {index_name} no existe en {keyspace}")

    # monotonic(): no le afecta un ajuste del reloj de pared durante la espera.
    deadline: float = time.monotonic() + timeout
    pending: list[str] = list(probes)  # si nunca llegamos a probar nada, están todos pendientes
    stalled_since: dict[str, float] = {}  # pendientes sin build en marcha, y desde cuándo
    recreated: set[str] = set()
    while True:
        build_running: bool | None = _background_build_running(session, keyspace, kind_markers)
        # `is not True` trata igual el False ("sabemos que no hay build en
        # marcha") y el None ("no sabemos, system_views no responde"): en
        # ambos casos merece la pena intentar la consulta real. Sólo cuando
        # la respuesta es un True confirmado nos ahorramos ese intento, que
        # sabemos que va a fallar con IndexNotAvailableException.
        if build_running is not True:
            pending = [
                name for name, probe_cql in probes.items() if not _probe_ok(session, probe_cql)
            ]
            if not pending:  # las consultas de prueba ya no fallan: hemos terminado
                return
            # Sin build en marcha (False confirmado) y la prueba sigue fallando.
            if recreate_indexes and build_running is False:
                now: float = time.monotonic()
                for name in pending:
                    if name not in recreated and now - stalled_since.setdefault(name, now) >= (
                        stall_timeout
                    ):
                        print(f"El build de {name} no terminó: se borra y se crea de nuevo.")
                        _recreate_index(session, keyspace, name)
                        recreated.add(name)
        else:
            stalled_since.clear()  # hay un build en marcha: se vuelve a empezar a contar
        # O bien el build sigue en marcha, o bien alguna prueba real todavía
        # falla: en los dos casos toca esperar un poco y volver a intentarlo.
        if time.monotonic() >= deadline:
            raise TimeoutError(
                f"{pending} de {keyspace} no estuvo(ieron) listo(s) en {timeout:.0f}s."
            )
        time.sleep(poll_interval)

In [ ]:
# Los `execute()` anteriores sólo esperaron a que el cambio de esquema se
# propagara; aquí se espera a que el build de cada índice haya terminado de verdad.
wait_for_schema_build(
    session,
    keyspace="stackoverflow",
    probes={
        "tags_tagname_sai": "SELECT Id FROM stackoverflow.Tags WHERE TagName = '.net'",
        "tags_count_sai": "SELECT Id FROM stackoverflow.Tags WHERE Count > 1000000 LIMIT 1",
    },
    # Los nombres de `probes` son los de los índices: si un build se aborta, se
    # vuelve a crear el índice.
    recreate_indexes=True,
)
print("Índices SAI de Tags listos.")

Se pueden hacer consultas «normales» usando los índices. Ésta busca una etiqueta por su nombre: `TagName` no es la clave de partición de `Tags` (lo es `Id`), así que sin el índice Cassandra rechazaría la consulta.

In [ ]:
result_dotnet: ResultSet = session.execute("""
    SELECT Id, TagName, Count
    FROM stackoverflow.Tags
    WHERE TagName = '.net'
""")
for row in result_dotnet:
    print(row.id, row.tagname, row.count)

Un índice SAI también resuelve rangos. Esta consulta pide las etiquetas usadas en más de 5 000 preguntas, estén en la partición que estén (cada etiqueta es una partición). Las filas no salen ordenadas por `Count`: el índice filtra, pero no ordena.

In [ ]:
# Rango sobre una columna numérica, en cualquier partición.
result_tags_more_5000: ResultSet = session.execute("""
    SELECT Id, TagName, Count
    FROM stackoverflow.Tags
    WHERE Count > 5000
""")
for row in result_tags_more_5000:
    print(row.id, row.tagname, row.count)

### Fin de la carga de `Posts`

Lo que sigue lee `Questions` y `Answers`, así que aquí esperamos a que termine la carga de `Posts`, si no lo ha hecho ya. La de `Users` empieza entonces y sigue en segundo plano mientras consultamos preguntas y respuestas: no la necesitamos hasta la unión de `Questions` y `Users`.

In [ ]:
# Esperar a que termine la carga de Posts (preguntas y respuestas). La de Users
# empieza entonces en segundo plano: se espera más abajo, antes de la unión.
import asyncio

_done: set[Task[dict[str, int]]]
pending: set[Task[dict[str, int]]]
_done, pending = await asyncio.wait((future_posts,), timeout=1800)

print_log(posts_log_queue)

if pending:
    raise TimeoutError("La carga de Posts no terminó a tiempo.")

# Al recoger el resultado se propagan los errores ocurridos en el hilo.
posts_inserted: dict[str, int] = await future_posts
print(posts_inserted)

Se muestra un ejemplo de consulta. Nótese que como las preguntas se buscan por `Id`, la clave de partición de `Questions` basta y no hace falta ningún índice adicional ni su almacenamiento.

In [ ]:
from cassandra.cluster import ResultSet

question_3_result: ResultSet = session.execute("""
    SELECT Id, CreationDate, Title, Body, AnswerCount
    FROM stackoverflow.Questions
    WHERE Id = 3
""")

for row in question_3_result:
    print(row.id, row.creationdate, row.title)
    print(f"Número de respuestas: {row.answercount}")
    print("Texto de la pregunta:")
    print(row.body)
    print("...")

Si antes vimos la pregunta 3, podemos ver sus respuestas. Con la clave elegida, todas las respuestas de una pregunta pertenecen a la misma partición (en la tabla `Answers`), así que se recuperan con una sola consulta. A una partición con muchas filas ordenadas por *clustering* se la llama *wide partition* (*wide row* en la terminología antigua).

In [ ]:
answers_3_result: ResultSet = session.execute("""
    SELECT ParentId, Id, CreationDate, Body, Score
    FROM stackoverflow.Answers
    WHERE ParentId = 3
""")
for row in answers_3_result:
    print(row.id, row.creationdate, row.parentid, row.score)
    print(row.body)
    print("...")

Con `Answers` ya cargada, creamos el último índice.

<details><summary>El siguiente índice permite buscar una respuesta por su `Id`, pero cada búsqueda pregunta a muchos nodos. Si esa consulta fuera frecuente, ¿cómo se haría en Cassandra para que tenga más rendimiento? <span style="color: #007acc;">▶️ Haz clic para ver la respuesta</span></summary>
Crear una tabla adicional cuya clave de partición sea la columna por la que se busca, de manera que la consulta lea una sola partición. Es lo que hace `TagsByName` más abajo con las etiquetas, y lo que pide el EJERCICIO 6 con `QuestionsByUser` para las preguntas de un autor.
</details>

In [ ]:
# `Id` no es la clave de partición de Answers (lo es `ParentId`): sin el índice,
# no se podría buscar una respuesta por su `Id`.
session.execute("""
    CREATE INDEX IF NOT EXISTS answers_id_sai
    ON stackoverflow.Answers (Id)
    USING 'sai'
""")

In [ ]:
wait_for_schema_build(
    session,
    keyspace="stackoverflow",
    probes={
        "answers_id_sai": "SELECT ParentId FROM stackoverflow.Answers WHERE Id = -1",
    },
    recreate_indexes=True,
    # El índice de Answers tarda lo que tarde el recorrido de sus SSTables, quizá con la
    # carga de Users todavía en marcha: el timeout por defecto (180 s) puede quedarse
    # corto en Colab o en un runner de CI compartido. Se amplía sólo para esta llamada.
    timeout=420.0,
)
print("Índice SAI de Answers listo.")

Esta consulta busca una respuesta por su `Id` (la 1006, una de las de la pregunta 1003): `Id` no es la clave de partición de `Answers` (lo es `ParentId`), así que sin el índice Cassandra rechazaría la consulta. Como el índice no dice en qué nodo está esa respuesta, puede tener que preguntar a muchos nodos para devolver una sola fila: sirve para ver que el índice funciona, no como diseño recomendable.

In [ ]:
result_answer_1006: ResultSet = session.execute("""
    SELECT Id, ParentId, CreationDate, Body
    FROM stackoverflow.Answers
    WHERE Id = 1006
""")
for row in result_answer_1006:
    print(row.id, row.parentid, row.creationdate)
    print("Texto de la respuesta:")
    print(row.body)
    print("...")

### `ALLOW FILTERING`

Un índice resuelve las condiciones sobre las columnas indexadas. Si se pone una condición sobre una columna que no está en la clave ni en un índice, Cassandra rechaza la consulta: tendría que **leer filas para después descartar las que no cumplen la condición**, y su coste dependería de cuántas filas lee, no de cuántas devuelve. `ALLOW FILTERING` es la cláusula con la que se acepta ese coste de forma explícita, y lo que cuesta depende de cuánto se haya acotado antes:

- **Dentro de una partición.** La primera consulta pide las respuestas de la pregunta 1003 con puntuación positiva: `ParentId` localiza la partición y `Score`, que no tiene índice en `Answers`, sólo descarta filas de esa partición. Es el uso razonable, porque el coste lo acota el tamaño de la partición.
- **En toda la tabla.** La segunda pone una condición sobre `WikiPostId`, una columna sin índice: el servidor recorre toda la tabla y filtra. En general no es recomendable (y en producción, nunca), porque su coste depende del tamaño de la tabla y no del resultado. Aquí lo toleramos porque `Tags` es pequeña; si la tabla creciera o el umbral lo diera el usuario, haría falta un índice o una tabla que sirviera esa consulta.

In [ ]:
# Sin ALLOW FILTERING, la consulta se rechaza. Se pone en un try/except para
# mostrar el error y ejecutar después la consulta correcta.
try:
    session.execute("""
        SELECT Id, Score
        FROM stackoverflow.Answers
        WHERE ParentId = 1003 AND Score > 0
    """)
# Se captura Exception, como en el EJERCICIO 3, para ver qué clase de error
# lanza el servidor.
except Exception as e:
    print(f"Error ({type(e).__name__}): {e}")

# Dentro de una partición: ParentId la localiza y Score sólo descarta sus filas.
result_answers_1003: ResultSet = session.execute("""
    SELECT Id, Score
    FROM stackoverflow.Answers
    WHERE ParentId = 1003 AND Score > 0
    ALLOW FILTERING
""")
for row in result_answers_1003:
    print(row.id, row.score)

# En toda la tabla: se recorren todas las particiones de Tags. WikiPostId, el
# Id de la publicación con la wiki de la etiqueta, no tiene índice.
result_tags_recent_wiki: ResultSet = session.execute("""
    SELECT Id, TagName, WikiPostId
    FROM stackoverflow.Tags
    WHERE WikiPostId > 620000
    ALLOW FILTERING
""")
for row in result_tags_recent_wiki:
    print(row.id, row.tagname, row.wikipostid)

### Qué mecanismo elegir

| Necesidad | Mecanismo que probar |
|---|---|
| Búsqueda ocasional por otra columna: igualdad o rangos | Índice SAI |
| Lectura frecuente por otra clave, con orden o `LIMIT` | Tabla diseñada para esa consulta |
| Filtrar dentro de una partición, o explorar una tabla pequeña | `ALLOW FILTERING`, con su coste de lectura y descarte |

El índice amplía lo que se puede consultar, pero no sustituye a una clave diseñada para la consulta. El EJERCICIO 6 de la sesión 6 los compara.

In [ ]:
# @title Diapositiva: Un índice no reemplaza el modelo
from IPython.display import HTML, display; display(HTML('<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#fff9db;color:#29261d;border:1px solid #dfcc78;border-top:10px solid #287681;border-radius:10px;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:22px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#287681" size="8" noshade style="display:none"><div style="color:#287681;font-size:15px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#287681" style="color:inherit"><b>Recapitulación</b></font></div><h1 style="color:#7b2c19;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:34px;line-height:1.15;margin:0 0 22px">Un índice no reemplaza el modelo</h1><ul style=\'margin:18px 0 8px;padding-left:30px\'><li style=\'margin:10px 0;list-style-position:outside\'>Un índice SAI permite buscar por otra columna, también por rango</li><li style=\'margin:10px 0;list-style-position:outside\'>Es local a cada nodo: sin clave de partición, la consulta pregunta a muchos</li><li style=\'margin:10px 0;list-style-position:outside\'>La construcción del índice ocurre después de la propagación del esquema</li><li style=\'margin:10px 0;list-style-position:outside\'><code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>ALLOW FILTERING</code> lee filas y las descarta: asumible en una partición, no en una tabla grande</li><li style=\'margin:10px 0;list-style-position:outside\'>Si una consulta frecuente cruza particiones, quizá falte una tabla derivada</li></ul><blockquote style=\'background:#fff0ae;color:inherit;border:none;border-left:6px solid #287681;margin:22px 0 0;padding:12px 16px\'><div style=\'margin-top:0px\'>En Cassandra la respuesta habitual a un JOIN es rediseñar la escritura</div></blockquote><hr color="#287681" size="2" noshade style="display:none"></div>'))

### Fin de la carga de `Users`

La unión que viene a continuación necesita también `Users`. Esperamos a que termine su carga, si no lo ha hecho ya, y consultamos un usuario por su clave, `Id`.

In [ ]:
# Esperar a que termine la carga de Users.
_done: set[Task[dict[str, int]]]
pending: set[Task[dict[str, int]]]
_done, pending = await asyncio.wait((future_users,), timeout=1800)

print_log(users_log_queue)

if pending:
    raise TimeoutError("La carga de Users no terminó a tiempo.")

# Al recoger el resultado se propagan los errores ocurridos en el hilo.
users_inserted: dict[str, int] = await future_users
print(users_inserted)

In [ ]:
users_result: ResultSet = session.execute("""
    SELECT Id, CreationDate, DisplayName, Reputation
    FROM stackoverflow.Users
    WHERE Id = 1
""")
for row in users_result:
    print(row.id, row.creationdate, row.displayname)
    print(f"Reputación: {row.reputation}")
    print("...")

In [ ]:
# @title Diapositiva: Resolver un JOIN por desnormalización
from IPython.display import HTML, display; display(HTML('<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#fff9db;color:#29261d;border:1px solid #dfcc78;border-top:10px solid #c69200;border-radius:10px;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:22px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#c69200" size="8" noshade style="display:none"><div style="color:#c69200;font-size:15px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#c69200" style="color:inherit"><b>A continuación</b></font></div><h1 style="color:#7b2c19;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:34px;line-height:1.15;margin:0 0 22px">Resolver un JOIN por desnormalización</h1><ul style=\'margin:18px 0 8px;padding-left:30px\'><li style=\'margin:10px 0;list-style-position:outside\'>Identificar la consulta que queremos servir</li><li style=\'margin:10px 0;list-style-position:outside\'>Crear una tabla con los datos ya combinados</li><li style=\'margin:10px 0;list-style-position:outside\'>Comparar varias estrategias de acceso y sus costes</li><li style=\'margin:10px 0;list-style-position:outside\'>Aceptar que una escritura puede tener que actualizar varias tablas</li></ul><blockquote style=\'background:#fff0ae;color:inherit;border:none;border-left:6px solid #c69200;margin:22px 0 0;padding:12px 16px\'><div style=\'margin-top:0px\'>La duplicación deja de ser un accidente y pasa a ser una decisión explícita</div></blockquote><hr color="#c69200" size="2" noshade style="display:none"></div>'))

### ¿Cómo hacemos un *JOIN* en Cassandra?

Respuesta rápida: **No se pueden hacer JOINs en Cassandra**. Para una lectura frecuente, la aplicación puede crear una tabla adicional que almacene los datos combinados. Las vistas materializadas (las vemos en la teoría) cambian el acceso a una tabla base, pero no unen tablas distintas. Por ahora, vamos a intentar hacerlo de forma manual con Python.

La consulta que haremos es: Mostrar el Id, el título, el nombre del usuario que hizo la pregunta, la fecha y los tags de la misma en un resultado. En SQL equivalente:

```sql
SELECT q.Id AS question_id, q.Title AS question_title,
       u.DisplayName AS user_name, q.OwnerUserId AS owner_user_id,
       q.CreationDate AS creation_date, q.Tags AS question_tags
FROM Questions q
JOIN Users u ON q.OwnerUserId = u.Id
```

Como se ve, el nombre del usuario no está en la tabla de preguntas, por lo que hay que hacer un *JOIN* para obtenerlo.

Esta tabla materializa una combinación de `Questions` y `Users` para una consulta concreta: las preguntas de un usuario, dado su nombre, ordenadas por fecha. Al escribir una pregunta nueva o cambiar un nombre, la aplicación tendría que actualizar también la copia. No es un `JOIN` ejecutado en cada lectura.

La clave primaria es `(user_name, creation_date, question_id)`. `user_name` es la clave de partición, así que las preguntas de un usuario quedan juntas. `creation_date` es la primera columna de *clustering*: las ordena dentro de la partición y permite pedir un rango de fechas. `question_id` va al final sólo para distinguir dos preguntas del mismo usuario en el mismo milisegundo, que en este *dump* existen (la 545165 y la 545166): sin él, la segunda escritura sustituiría en silencio a la primera.

Primero creamos la tabla destino que va a llevar los datos "unidos".

La celda empieza con `CREATE TABLE IF NOT EXISTS` y sigue con `TRUNCATE`, que vacía la tabla y conserva su definición. Las dos órdenes sólo sirven para que la celda sea reproducible: si se ejecuta otra vez, parte de la misma tabla vacía. No se borra la tabla con `DROP TABLE` para crearla otra vez justo después: un cambio de esquema llega a cada nodo por separado, y la tabla recreada es otra para Cassandra (tiene otro identificador interno, aunque se llame igual). Si se borra y se crea tan rápido, una escritura inmediata puede llegar a un nodo que todavía tiene la tabla anterior, o ninguna, y ese nodo la rechaza con `INCOMPATIBLE_SCHEMA`. El *driver* espera a que todos los nodos tengan la misma versión del esquema tras cada cambio, pero sólo hasta `max_schema_agreement_wait` (30 s, fijado al crear el `Cluster`).


In [ ]:
session.execute("""
    CREATE TABLE IF NOT EXISTS stackoverflow.QuestionsWithUserNames (
        question_id bigint,
        question_title text,
        user_name text,
        owner_user_id bigint,
        creation_date timestamp,
        question_tags text,
        PRIMARY KEY (user_name, creation_date, question_id)
    ) WITH compression = {'class': 'LZ4Compressor'};
""")
session.execute("TRUNCATE stackoverflow.QuestionsWithUserNames")

Vamos a empezar con la consulta de las preguntas.


In [ ]:
from cassandra.query import PreparedStatement

q_u_questions_select_stmt: PreparedStatement = session.prepare("""
    SELECT Id, Title, OwnerUserId, CreationDate, Tags
    FROM stackoverflow.Questions
    """)

Después la consulta sobre cada usuario. Para evitar hacer muchas consultas, almacenamos los usuarios ya consultados en una caché. Como veremos, esta solución no es muy buena, pero muestra cómo hacerlo de forma flexible con python con varias opciones.

In [ ]:
from functools import lru_cache

q_u_users_select_stmt: PreparedStatement = session.prepare("""
    SELECT DisplayName
    FROM stackoverflow.Users
    WHERE Id = ?
    """)


@lru_cache(maxsize=1_000_000)
def get_user(user_id: int) -> str | None:
    row: ResultSet = session.execute(q_u_users_select_stmt, [user_id]).one()
    return row.displayname if row else None

También necesitamos el *statement* que incluye la inserción de los datos en la tabla `QuestionsWithUserNames`. Este sería un ejemplo de cómo podría ser:


In [ ]:
insert_question_with_username_stmt: PreparedStatement = session.prepare("""
    INSERT INTO stackoverflow.QuestionsWithUserNames (question_id, question_title, user_name, owner_user_id, creation_date, question_tags)
    VALUES (?, ?, ?, ?, ?, ?)
    """)

Insertamos en `QuestionsWithUserNames` con `populate_table()`. La función recibe un `ResultSet` de `Questions` y lo recorre fila a fila: no carga toda la tabla en memoria. `q_u_tuple_from_question()` transforma cada fila en los valores del `INSERT` y omite las preguntas sin autor, pues `user_name` forma parte de la clave.

Para no alargar la sesión, la tabla sólo guarda las preguntas **anteriores a 2020** (`FIN_MUESTRA`): unas 97 000 de las 194 000 que tienen autor. Basta para comparar las dos formas de construirla y para las consultas de más abajo. `CreationDate` no es la clave de partición de `Questions`, así que el `WHERE` no puede filtrar por ella sin `ALLOW FILTERING`: se recorre la tabla entera y `q_u_tuple_from_question()` descarta también las preguntas posteriores. Leer una pregunta cuesta mucho menos que escribirla y, en la versión 1, que buscar su autor.

Medimos el tiempo de recorrer las preguntas, consultar los autores e insertar las filas. Antes vaciamos la caché de autores para que una reejecución no aproveche la anterior.

Usamos `LOAD_PROFILE` porque se trata de una carga masiva. Su consistencia `ANY` no garantiza una lectura inmediata; el perfil por omisión usa `LOCAL_QUORUM` para las operaciones interactivas.


In [ ]:
from datetime import datetime
from typing import Any

# La tabla sólo guarda las preguntas anteriores a esta fecha: unas 97 000 con autor.
FIN_MUESTRA: datetime = datetime(2020, 1, 1)


def q_u_tuple_from_question(question: Any) -> tuple | None:
    """Valores del INSERT de QuestionsWithUserNames para una fila de Questions."""
    if not question.owneruserid:
        return None  # sin autor no hay user_name, que es la clave de partición
    if question.creationdate >= FIN_MUESTRA:
        return None  # fuera de la muestra
    # Si el autor ya no está en Users, se guarda con un nombre fijo: la clave de
    # partición no admite null.
    user_name: str = get_user(question.owneruserid) or "(borrado)"
    return (
        question.id,
        question.title,
        user_name,
        question.owneruserid,
        question.creationdate,
        question.tags,
    )

In [ ]:
from time import perf_counter

# Empezar con la caché vacía también al reejecutar esta celda.
get_user.cache_clear()
time_before_v1: float = perf_counter()

# Insertar preguntas con nombres de usuario en la nueva tabla, página a página
q_u_inserted_v1: int = populate_table(
    session,
    insert_question_with_username_stmt,
    session.execute(q_u_questions_select_stmt),
    to_values=q_u_tuple_from_question,
    # Perfil de carga: escritura masiva, consistencia ANY y timeout largo.
    execution_profile=LOAD_PROFILE,
)
time_v1: float = perf_counter() - time_before_v1
print(f"Preguntas insertadas en QuestionsWithUserNames: {q_u_inserted_v1}")
print(f"Tiempo total de la versión 1 (consultas con caché): {time_v1:.2f} segundos.")

Vamos a hacer una consulta para ver los datos insertados. Accedemos por la clave primaria: primero, todas las preguntas de un usuario que hay en la tabla (las anteriores a 2020), que salen ordenadas por fecha; después, sólo las de un año, con un rango sobre `creation_date`.


In [ ]:
session.execute("""
    SELECT question_id, question_title, user_name, owner_user_id, creation_date, question_tags
    FROM stackoverflow.QuestionsWithUserNames
    WHERE user_name = 'Jorgesys';
""").all()

In [ ]:
session.execute("""
    SELECT question_id, question_title, user_name, owner_user_id, creation_date, question_tags
    FROM stackoverflow.QuestionsWithUserNames
    WHERE user_name = 'Jorgesys'
      AND creation_date >= '2016-01-01' AND creation_date < '2017-01-01';
""").all()

In [ ]:
# @title Diapositiva: Tres formas de construir la tabla
from IPython.display import HTML, display; display(HTML('<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#fff9db;color:#29261d;border:1px solid #dfcc78;border-top:10px solid #c69200;border-radius:10px;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:22px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#c69200" size="8" noshade style="display:none"><div style="color:#c69200;font-size:15px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#c69200" style="color:inherit"><b>A continuación</b></font></div><h1 style="color:#7b2c19;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:34px;line-height:1.15;margin:0 0 22px">Tres formas de construir la tabla</h1><ul style=\'margin:18px 0 8px;padding-left:30px\'><li style=\'margin:10px 0;list-style-position:outside\'>V1: consultar el usuario de cada pregunta, con una caché LRU</li><li style=\'margin:10px 0;list-style-position:outside\'>V2: cargar los usuarios en un diccionario y recorrer después las preguntas</li><li style=\'margin:10px 0;list-style-position:outside\'>V3: solapar las lecturas, a cambio de más memoria y complejidad</li><li style=\'margin:10px 0;list-style-position:outside\'>Una lectura frecuente se sirve desde la tabla ya desnormalizada</li></ul><blockquote style=\'background:#fff0ae;color:inherit;border:none;border-left:6px solid #c69200;margin:22px 0 0;padding:12px 16px\'><div style=\'margin-top:0px\'>La aplicación debe mantener la copia cuando cambian los datos de origen</div></blockquote><hr color="#c69200" size="2" noshade style="display:none"></div>'))

### Comparación de las tres estrategias de unión

Hay tres formas de construir las mismas tuplas: la 1 es la que acabamos de ejecutar, la 2 es la que ejecutaremos a continuación y la 3 lanzaría los dos recorridos a la vez, en hilos separados (`asyncio.to_thread()` y `gather()`), y combinaría al final.

| Versión | Lecturas | Coste que domina | Compromiso |
|---|---|---|---|
| 1. Usuario por pregunta | Recorrido de `Questions` y una consulta por autor no cacheado | Latencia de muchas consultas | Código simple; la caché evita algunas consultas repetidas |
| 2. Diccionario de usuarios | Recorrido de `Users`, después de `Questions` | Dos recorridos secuenciales | Pocas consultas; guarda todos los usuarios en memoria. Puede que no quepan |
| 3. Lecturas en paralelo | Recorridos de `Users` y `Questions` solapados | El recorrido más lento, más la combinación final | Menor espera potencial; código más complejo y memoria para ambas entradas |

Medimos las dos primeras en esta ejecución, incluyendo sus lecturas y escrituras, pero sin cronometrar la preparación de los *statements* ni el vaciado de la tabla. La segunda se ejecuta después de la primera y puede beneficiarse de las cachés de Cassandra y del sistema. Para una comparación rigurosa habría que repetir las medidas en distintas condiciones y comparar medianas: un tiempo aislado depende de la máquina, la carga y las cachés. La decisión de modelado es anterior: si esta lectura es frecuente, una tabla preparada para ella evita rehacer la unión en cada consulta. Una vista materializada cambia la clave de acceso a **una tabla base**; no une automáticamente `Questions` con `Users`.


En la primera versión consultamos `Users` por cada autor que no está en la caché LRU: pueden ser muchas consultas pequeñas. Ahora leemos **todos** los usuarios una sola vez, guardamos su nombre en un diccionario y después recorremos las preguntas consultando ese diccionario. Hay menos consultas a cambio de tener todos los usuarios en memoria.

Volvemos a poblar la misma tabla desde cero con esta segunda estrategia. El `TRUNCATE` elimina las filas de la primera versión, pero conserva la definición y el *statement* preparado; así no mezclamos los resultados.

In [ ]:
all_users_stmt: PreparedStatement = session.prepare("""
    SELECT Id, DisplayName
    FROM stackoverflow.Users
    """)
time_before_v2_1: float = perf_counter()
all_users_result_set: ResultSet = session.execute(all_users_stmt)
user_id_to_name: dict[int, str] = {row.id: row.displayname for row in all_users_result_set}
time_after_v2_1: float = perf_counter()

print(f"Tiempo para construir user_id_to_name: {time_after_v2_1 - time_before_v2_1:.2f} segundos.")

In [ ]:
from typing import Any


def q_u_tuple_from_question_v2(question: Any) -> tuple | None:
    """Valores del INSERT de QuestionsWithUserNames, con el nombre sacado del diccionario."""
    if not question.owneruserid:
        return None  # sin autor no hay user_name, que es la clave de partición
    if question.creationdate >= FIN_MUESTRA:
        return None  # fuera de la muestra, como en la versión 1
    # Si el autor ya no está en Users, se guarda con un nombre fijo: la clave de
    # partición no admite null.
    user_name: str = user_id_to_name.get(question.owneruserid) or "(borrado)"
    return (
        question.id,
        question.title,
        user_name,
        question.owneruserid,
        question.creationdate,
        question.tags,
    )

In [ ]:
session.execute("TRUNCATE stackoverflow.QuestionsWithUserNames")

# Insertar preguntas con nombres de usuario en la nueva tabla, página a página:
# cada tupla se construye al enviarla, sin guardar todas las preguntas.
time_before_v2_2: float = perf_counter()
q_u_inserted: int = populate_table(
    session,
    insert_question_with_username_stmt,
    session.execute(q_u_questions_select_stmt),
    to_values=q_u_tuple_from_question_v2,
    # Perfil de carga: escritura masiva, consistencia ANY y timeout largo.
    execution_profile=LOAD_PROFILE,
)
time_after_v2_2: float = perf_counter()
print(f"Preguntas insertadas en QuestionsWithUserNames: {q_u_inserted}")
assert q_u_inserted == q_u_inserted_v1
time_v2: float = time_after_v2_1 - time_before_v2_1 + time_after_v2_2 - time_before_v2_2
print(f"Tiempo para escribir las preguntas: {time_after_v2_2 - time_before_v2_2:.2f} segundos.")
print(f"Versión 1 (consultas con caché): {time_v1:.2f} segundos.")
print(f"Versión 2 (diccionario de usuarios): {time_v2:.2f} segundos.")

In [ ]:
import gc

# El diccionario de usuarios ya no hace falta. También limpiamos la caché de get_user() para liberar memoria
get_user.cache_clear()
del user_id_to_name
gc.collect()

In [ ]:
session.execute("""
    SELECT question_id, question_title, user_name, owner_user_id, creation_date, question_tags
    FROM stackoverflow.QuestionsWithUserNames
    WHERE user_name = 'Jorgesys';
""").all()

In [ ]:
# @title Diapositiva: Diseñar la tabla para la consulta
from IPython.display import HTML, display; display(HTML('<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#fff9db;color:#29261d;border:1px solid #dfcc78;border-top:10px solid #287681;border-radius:10px;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:22px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#287681" size="8" noshade style="display:none"><div style="color:#287681;font-size:15px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#287681" style="color:inherit"><b>Recapitulación</b></font></div><h1 style="color:#7b2c19;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:34px;line-height:1.15;margin:0 0 22px">Diseñar la tabla para la consulta</h1><ul style=\'margin:18px 0 8px;padding-left:30px\'><li style=\'margin:10px 0;list-style-position:outside\'>Consultar cada autor puede suponer muchos viajes al servidor</li><li style=\'margin:10px 0;list-style-position:outside\'>Un diccionario reduce los viajes a cambio de memoria</li><li style=\'margin:10px 0;list-style-position:outside\'>Construir la tabla una vez evita rehacer la unión en cada lectura</li><li style=\'margin:10px 0;list-style-position:outside\'>La aplicación debe mantener la copia al cambiar preguntas o usuarios</li></ul><blockquote style=\'background:#fff0ae;color:inherit;border:none;border-left:6px solid #287681;margin:22px 0 0;padding:12px 16px\'><div style=\'margin-top:0px\'>La copia puede mantenerla la aplicación o, en una vista materializada, Cassandra</div></blockquote><hr color="#287681" size="2" noshade style="display:none"></div>'))

In [ ]:
# @title Diapositiva: Para el examen: desnormalizar en vez de unir
from IPython.display import HTML, display; display(HTML('<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#fff9db;color:#29261d;border:1px solid #dfcc78;border-top:10px solid #8a6a00;border-radius:10px;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:22px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#8a6a00" size="8" noshade style="display:none"><div style="color:#8a6a00;font-size:15px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#8a6a00" style="color:inherit"><b>Para el examen</b></font></div><h1 style="color:#7b2c19;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:34px;line-height:1.15;margin:0 0 22px">Para el examen: desnormalizar en vez de unir</h1><ul style=\'margin:18px 0 8px;padding-left:30px\'><li style=\'margin:10px 0;list-style-position:outside\'>Cassandra no hace JOIN: la unión se resuelve al escribir, no al leer</li><li style=\'margin:10px 0;list-style-position:outside\'>La misma entidad vive en varias tablas, una por patrón de acceso</li><li style=\'margin:10px 0;list-style-position:outside\'>Duplicar obliga a decidir quién mantiene la copia y qué pasa si falla a medias</li><li style=\'margin:10px 0;list-style-position:outside\'>El coste se traslada de la lectura a la escritura y al espacio ocupado</li><li style=\'margin:10px 0;list-style-position:outside\'>Leer una partición completa por su clave es lo que hace escalable el modelo</li></ul><blockquote style=\'background:#fff0ae;color:inherit;border:none;border-left:6px solid #8a6a00;margin:22px 0 0;padding:12px 16px\'><div style=\'margin-top:0px\'>Pregunta típica: convertir una consulta SQL con JOIN en tablas de Cassandra</div></blockquote><hr color="#8a6a00" size="2" noshade style="display:none"></div>'))

### Otro ejemplo de `populate_table()`: una tabla para buscar etiquetas por nombre

Más arriba creamos un índice SAI sobre `Tags (TagName)` para buscar una etiqueta por su nombre. La alternativa propia de Cassandra es una tabla cuya clave de partición sea justo ese nombre: `TagsByName`, con una fila por etiqueta. Se puebla desde `Tags` con los mismos pasos que seguirán los ejercicios que crean tablas:

1. crear la tabla si no existe y vaciarla con `TRUNCATE`, para poder reejecutar la celda (los ejercicios empiezan en cambio con `DROP TABLE IF EXISTS`: al probar otra clave hay que recrear la tabla, y `CREATE TABLE IF NOT EXISTS` conservaría sin avisar la definición anterior);
2. preparar el `INSERT`;
3. llamar a `populate_table()` con el `ResultSet` del `SELECT` de origen.

Aquí no hace falta `to_values`: el `SELECT` pide las columnas (`TagName`, `Id`, `Count`) en el mismo orden que los `?` del `INSERT`, y `populate_table()` envía cada fila tal cual. `to_values` sólo se necesita cuando la fila de origen no basta por sí misma: cuando hay que calcular un valor que no está en la tabla de origen (como el nombre del autor en `QuestionsWithUserNames`) o transformar uno que sí está (por ejemplo, convertir un texto en un `set`), o cuando hay que descartar filas que el `WHERE` no puede filtrar (como las preguntas sin autor, que se quedarían sin clave de partición, o las posteriores a `FIN_MUESTRA`). Y si los valores se calculan antes en Python (por ejemplo, agrupando en un diccionario), a `populate_table()` se le pasan directamente las tuplas, en una lista o en cualquier otro iterable: el cargador de los Parquet le pasa un `zip()`.

In [ ]:
session.execute("""
    CREATE TABLE IF NOT EXISTS stackoverflow.TagsByName (
        tag_name text PRIMARY KEY,
        tag_id bigint,
        question_count int
    );
""")
session.execute("TRUNCATE stackoverflow.TagsByName")
insert_tag_by_name_stmt: PreparedStatement = session.prepare(
    "INSERT INTO stackoverflow.TagsByName (tag_name, tag_id, question_count) VALUES (?, ?, ?)"
)

tags_by_name_written: int = populate_table(
    session,
    insert_tag_by_name_stmt,
    # Las columnas, en el orden de los `?` del INSERT: no hace falta to_values.
    session.execute("SELECT TagName, Id, Count FROM stackoverflow.Tags;"),
    # Perfil de carga: escritura masiva, consistencia ANY y timeout largo.
    execution_profile=LOAD_PROFILE,
)
print(f"Etiquetas escritas en TagsByName: {tags_by_name_written}")

La misma búsqueda por nombre que antes servía el índice es ahora una consulta por la clave de partición de `TagsByName`: va directamente a los nodos que guardan esa partición. El precio es mantener una copia más de cada etiqueta.

In [ ]:
session.execute("""
    SELECT tag_name, tag_id, question_count
    FROM stackoverflow.TagsByName
    WHERE tag_name = '.net';
""").one()

In [ ]:
# @title Diapositiva: Cassandra obliga a explicitar el acceso
from IPython.display import HTML, display; display(HTML('<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#fff9db;color:#29261d;border:1px solid #dfcc78;border-top:10px solid #287681;border-radius:10px;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:22px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#287681" size="8" noshade style="display:none"><div style="color:#287681;font-size:15px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#287681" style="color:inherit"><b>Recapitulación</b></font></div><h1 style="color:#7b2c19;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:34px;line-height:1.15;margin:0 0 22px">Cassandra obliga a explicitar el acceso</h1><ul style=\'margin:18px 0 8px;padding-left:30px\'><li style=\'margin:10px 0;list-style-position:outside\'>La topología y los perfiles influyen en dónde se ejecuta una petición</li><li style=\'margin:10px 0;list-style-position:outside\'>La clave primaria decide partición y orden, no sólo identidad</li><li style=\'margin:10px 0;list-style-position:outside\'>Cargar, indexar y filtrar son elecciones con límites observables</li><li style=\'margin:10px 0;list-style-position:outside\'>Las tablas derivadas sustituyen a los JOIN y mantienen una lectura rápida</li></ul><blockquote style=\'background:#fff0ae;color:inherit;border:none;border-left:6px solid #287681;margin:22px 0 0;padding:12px 16px\'><div style=\'margin-top:0px\'>La última comprobación es diseñar una solución para una consulta concreta</div></blockquote><hr color="#287681" size="2" noshade style="display:none"></div>'))

In [ ]:
# @title Diapositiva: Consultas por clave y CQL que aún no había aparecido
from IPython.display import HTML, display; display(HTML('<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#fff9db;color:#29261d;border:1px solid #dfcc78;border-top:10px solid #c69200;border-radius:10px;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:22px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#c69200" size="8" noshade style="display:none"><div style="color:#c69200;font-size:15px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#c69200" style="color:inherit"><b>A continuación</b></font></div><h1 style="color:#7b2c19;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:34px;line-height:1.15;margin:0 0 22px">Consultas por clave y CQL que aún no había aparecido</h1><ul style=\'margin:18px 0 8px;padding-left:30px\'><li style=\'margin:10px 0;list-style-position:outside\'>Preguntar a las tablas cargadas: sólo por la clave declarada (o un índice)</li><li style=\'margin:10px 0;list-style-position:outside\'>Colecciones: <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>set</code>, <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>list</code> y <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>map</code></li><li style=\'margin:10px 0;list-style-position:outside\'>Niveles de consistencia en una lectura, y lo que cuestan</li><li style=\'margin:10px 0;list-style-position:outside\'>El tipo <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>COUNTER</code>, en una tabla aparte</li></ul><hr color="#c69200" size="2" noshade style="display:none"></div>'))

## Practicar CQL básico

Unas preguntas sencillas sobre las tablas que ya hemos cargado y unos bloques pequeños sobre construcciones de CQL que no han aparecido todavía: colecciones, niveles de consistencia y `COUNTER`. Terminamos con un ejercicio de diseño.

Cuando un enunciado pida explicar o interpretar algo, escríbelo como comentario en la propia celda de la solución, antes o después del código. Nadie lo corrige de forma automática, pero es lo que tendrás que saber defender sin el cuaderno.

### Consultas sobre las tablas cargadas

Recuerda las claves primarias de las tablas que hemos cargado:

| Tabla | Clave de partición | *Clustering* |
|---|---|---|
| `Tags` | `Id` | — |
| `Users` | `Id` | — |
| `Questions` | `Id` | `CreationDate` |
| `Answers` | `ParentId` | `CreationDate` |

Además están los índices SAI creados más arriba: `Tags (TagName)`, `Tags (Count)` y `Answers (Id)`.

Sin `ALLOW FILTERING`, una consulta sólo puede filtrar por la clave de partición completa (y, dentro de esa partición, por las columnas de *clustering* en su orden) o por una columna con índice. En cada ejercicio, fíjate en qué columnas usas en el `WHERE`.

Una consulta **sin** `WHERE` no filtra nada, así que tampoco pide `ALLOW FILTERING`: `SELECT … LIMIT n` lee las primeras filas en orden de *token* y se detiene, y `SELECT COUNT(*)` recorre la tabla entera. Las comprobaciones las usan para tomar una muestra o para contar, pero no sirven para buscar: las filas que devuelve `LIMIT` son unas cualesquiera.

### EJERCICIO 1: el autor de una pregunta

¿Quién escribió la pregunta 1003? Busca en `Questions` su `OwnerUserId` y, con él, el `DisplayName` del usuario en `Users`. Cada consulta debe filtrar sólo por la clave de partición de su tabla. Guarda el identificador del autor en `autor_1003_id` y su nombre en `autor_1003`.

In [ ]:
# Comprobación del EJERCICIO 1: el autor de una pregunta
assert "autor_1003_id" in globals(), "Guarda el identificador del autor en autor_1003_id"
assert "autor_1003" in globals(), "Guarda el nombre del autor en autor_1003"

# Camino paralelo, por otra tabla: QuestionsWithUserNames se particiona por el
# nombre del autor y guarda también su identificador. Si el nombre es el
# correcto, la pregunta 1003 está en su partición, con el identificador del autor.
preguntas_del_autor: dict[int, int] = {
    fila.question_id: fila.owner_user_id
    for fila in session.execute(
        "SELECT question_id, owner_user_id FROM stackoverflow.QuestionsWithUserNames "
        "WHERE user_name = %s;",
        (autor_1003,),
    )
}
print(f"Preguntas de {autor_1003!r} en QuestionsWithUserNames: {len(preguntas_del_autor)}")
assert 1003 in preguntas_del_autor, "La pregunta 1003 no es de un usuario con ese nombre"
assert preguntas_del_autor[1003] == autor_1003_id, "Ese identificador no es el del autor"
print("EJERCICIO 1: comprobado")

### EJERCICIO 2: las respuestas de una pregunta

¿Cuántas respuestas tiene la pregunta 1003 y cuál fue la primera? En `Answers`, las respuestas de una pregunta forman una sola partición (`ParentId`), ordenada por `CreationDate`: basta con una consulta a esa partición, sin ordenar en el cliente. Guarda el número de respuestas en `num_respuestas_1003` y el `Id` de la primera en `primera_respuesta_1003`.

In [ ]:
# Comprobación del EJERCICIO 2: las respuestas de una pregunta
from typing import Any

assert "num_respuestas_1003" in globals(), "Guarda el número de respuestas en num_respuestas_1003"
assert "primera_respuesta_1003" in globals(), "Guarda el Id de la primera en primera_respuesta_1003"

# Camino paralelo: el número de respuestas que declara la propia pregunta y,
# para la primera, una consulta de rango sobre el clustering: dentro de la
# partición 1003 no hay ninguna respuesta anterior a ella.
answer_count: int = (
    session.execute("SELECT AnswerCount FROM stackoverflow.Questions WHERE Id = 1003;")
    .one()
    .answercount
)
fila_primera: Any = session.execute(
    "SELECT ParentId, CreationDate FROM stackoverflow.Answers WHERE Id = %s;",
    (primera_respuesta_1003,),
).one()
assert fila_primera is not None, f"No existe la respuesta {primera_respuesta_1003}"
anteriores: int = (
    session.execute(
        "SELECT COUNT(*) AS total FROM stackoverflow.Answers "
        "WHERE ParentId = 1003 AND CreationDate < %s;",
        (fila_primera.creationdate,),
    )
    .one()
    .total
)
print(f"Tuyo: {num_respuestas_1003} respuestas | AnswerCount de la pregunta: {answer_count}")
print(f"Respuestas anteriores a la {primera_respuesta_1003}: {anteriores}")
assert num_respuestas_1003 == answer_count, "No coincide con el AnswerCount de la pregunta"
assert fila_primera.parentid == 1003, "Esa respuesta no es de la pregunta 1003"
assert anteriores == 0, "Hay respuestas a la pregunta 1003 anteriores a la guardada"
print("EJERCICIO 2: comprobado")

### EJERCICIO 3: lo que no se puede preguntar

Intenta ahora dos consultas sobre `Answers` sin `ALLOW FILTERING`, cada una dentro de un `try`/`except Exception as exc`:

1. Las respuestas escritas desde el 1 de enero de 2016, filtrando sólo por `CreationDate`. Guarda `type(exc).__name__` en `error_por_fecha`.
2. Las respuestas escritas por el autor de la pregunta 1003 (`OwnerUserId`, con el `autor_1003_id` del EJERCICIO 1). Guarda `type(exc).__name__` en `error_por_autor`.

Repite después la segunda con `ALLOW FILTERING` y guarda en `respuestas_autor_1003` la lista de los `Id` de esas respuestas.

Explica en un comentario por qué fallan las dos consultas aunque `CreationDate` forme parte de la clave primaria, qué hace el servidor para responder con `ALLOW FILTERING` y qué tabla diseñarías para responder la segunda pregunta sin él.

In [ ]:
# Comprobación del EJERCICIO 3: lo que no se puede preguntar
from typing import Any

assert "error_por_fecha" in globals(), "Guarda en error_por_fecha la clase de la excepción"
assert "error_por_autor" in globals(), "Guarda en error_por_autor la clase de la excepción"
assert "respuestas_autor_1003" in globals(), "Guarda en respuestas_autor_1003 la lista de Id"
print(f"Por fecha: {error_por_fecha} | por autor: {error_por_autor}")
assert error_por_fecha == "InvalidRequest", (
    "Filtrar sólo por la columna de clustering, sin la partición, debe rechazarse"
)
assert error_por_autor == "InvalidRequest", (
    "Filtrar por una columna sin clave ni índice debe rechazarse"
)
assert respuestas_autor_1003, "El autor de la pregunta 1003 también ha escrito respuestas"
assert len(set(respuestas_autor_1003)) == len(respuestas_autor_1003), "Hay Id repetidos"

# Camino paralelo: cada respuesta se revalida por el índice de Answers (Id).
for respuesta_id in respuestas_autor_1003:
    fila: Any = session.execute(
        "SELECT OwnerUserId FROM stackoverflow.Answers WHERE Id = %s;", (respuesta_id,)
    ).one()
    assert fila is not None, f"No existe la respuesta {respuesta_id}"
    assert fila.owneruserid == autor_1003_id, f"La respuesta {respuesta_id} es de otro autor"
print(f"Respuestas revalidadas: {len(respuestas_autor_1003)}")
print("EJERCICIO 3: comprobado")

### Tipos colección: `set`, `list` y `map`

Cassandra admite columnas de colección además de los tipos escalares:

- `set<T>`: conjunto sin repetidos, que Cassandra devuelve ordenado. Añadir un elemento que ya está no cambia nada.
- `list<T>`: secuencia ordenada por posición, que admite repetidos. Añadir un elemento ya presente lo repite, y reescribir una posición (`col[i] = ...`) obliga al servidor a leer antes la lista.
- `map<K, V>`: pares clave-valor; cada clave se escribe por separado.

La sintaxis de `list` y `map` es parecida a la de `set`. Sólo como referencia, sin ejecutar:

```sql
ALTER TABLE mi_tabla ADD historial list<text>;
UPDATE mi_tabla SET historial = historial + ['editado'] WHERE id = 1;

ALTER TABLE mi_tabla ADD propiedades map<text, text>;
UPDATE mi_tabla SET propiedades['idioma'] = 'es' WHERE id = 1;
```

En los datos de Stack Overflow, las etiquetas de una pregunta son un conjunto: no se repiten y su orden no significa nada. `Questions` las guarda en `Tags` como un texto con la forma `<etiqueta1><etiqueta2>`. Añadimos a `Questions` una columna `tag_set set<text>` con un `ALTER TABLE`: en Cassandra añadir una columna sólo cambia el esquema y no reescribe las filas existentes, que simplemente no tienen valor en ella hasta que se escribe. La rellenamos para la pregunta 1003. La actualización da la clave primaria completa (`Id` y `CreationDate`) y usa `tag_set = tag_set + ?`, que añade elementos sin sustituir el conjunto: asignarlo entero escribiría antes una *tombstone* que borra el contenido anterior.

In [ ]:
import re
from typing import Any

session.execute("ALTER TABLE stackoverflow.Questions ADD IF NOT EXISTS tag_set set<text>;")

pregunta_1003: Any = session.execute(
    "SELECT Id, CreationDate, Tags FROM stackoverflow.Questions WHERE Id = 1003;"
).one()
etiquetas_1003: set[str] = set(re.findall(r"<(.*?)>", pregunta_1003.tags))
print(f"Tags: {pregunta_1003.tags}")

add_tags_stmt: PreparedStatement = session.prepare(
    "UPDATE stackoverflow.Questions SET tag_set = tag_set + ? WHERE Id = ? AND CreationDate = ?"
)
session.execute(add_tags_stmt, (etiquetas_1003, 1003, pregunta_1003.creationdate))
session.execute("SELECT tag_set FROM stackoverflow.Questions WHERE Id = 1003;").one()

In [ ]:
# Volver a añadir una etiqueta que ya está no cambia el conjunto.
session.execute(add_tags_stmt, ({min(etiquetas_1003)}, 1003, pregunta_1003.creationdate))
session.execute("SELECT tag_set FROM stackoverflow.Questions WHERE Id = 1003;").one()

### EJERCICIO 4: las etiquetas de cada pregunta en un `set`

Rellena `tag_set` en las **20 000 primeras filas** de `Questions`, las que devuelve `SELECT Id, CreationDate, Tags FROM stackoverflow.Questions LIMIT 20000`, con el conjunto de etiquetas que guarda como texto su columna `Tags`, igual que en el ejemplo con la pregunta 1003. Esa muestra basta para ver cómo se comporta un `set`: escribir todas las preguntas no enseña nada más y alarga la sesión. Usa el *statement* preparado `add_tags_stmt` y `populate_table()` con el perfil de carga (`LOAD_PROFILE`): pásale el `ResultSet` de un `SELECT` sobre `Questions` y una función `to_values` que convierta cada pregunta en los valores de `add_tags_stmt`, como en la inserción de `QuestionsWithUserNames` y en el ejemplo de `TagsByName`. Guarda en la variable `preguntas_con_tag_set` el número de preguntas cuyo `tag_set` has escrito.

Explica después, en un comentario, qué pasaría al reejecutar la celda si `tag_set` fuera una `list<text>`.

In [ ]:
# Comprobación del EJERCICIO 4: las etiquetas de cada pregunta en un `set`
from typing import Any

assert "preguntas_con_tag_set" in globals(), (
    "Guarda en preguntas_con_tag_set el número de preguntas actualizadas"
)
# La muestra del enunciado son las 20 000 primeras preguntas. Se cuenta en el cliente: un
# COUNT(*) tiene un único plazo para todas sus páginas internas (range_request_timeout, 10 s).
preguntas_muestra: int = sum(
    1 for _ in session.execute("SELECT Id FROM stackoverflow.Questions LIMIT 20000;")
)
print(
    f"Preguntas con tag_set: {preguntas_con_tag_set} | preguntas de la muestra: {preguntas_muestra}"
)
assert preguntas_con_tag_set == preguntas_muestra, (
    "Todas las preguntas tienen etiquetas: hay que escribir tag_set en las 20 000 primeras"
)

# Camino paralelo, sobre una muestra y sin repetir la expresión regular: cada
# etiqueta del conjunto aparece entre < y > en Tags, y hay tantas como '<'.
# Son las 200 primeras filas del mismo recorrido, es decir, están dentro de las 20 000.
muestra: list[Any] = session.execute(
    "SELECT Id, Tags, tag_set FROM stackoverflow.Questions LIMIT 200;"
).all()
for fila in muestra:
    assert fila.tag_set is not None, f"La pregunta {fila.id} no tiene tag_set"
    assert len(fila.tag_set) == fila.tags.count("<"), (
        f"La pregunta {fila.id} no tiene tantas etiquetas en tag_set como en Tags"
    )
    assert all(f"<{etiqueta}>" in fila.tags for etiqueta in fila.tag_set), (
        f"tag_set de la pregunta {fila.id} tiene etiquetas que no están en Tags"
    )
print(f"Muestra revisada: {len(muestra)} preguntas")
print("EJERCICIO 4: comprobado")

### Niveles de consistencia en una lectura

In [ ]:
from cassandra.query import SimpleStatement

result_one: ResultSet = session.execute(
    SimpleStatement(
        "SELECT Id, DisplayName FROM stackoverflow.Users LIMIT 1;",
        consistency_level=ConsistencyLevel.ONE,
    )
)
result_quorum: ResultSet = session.execute(
    SimpleStatement(
        "SELECT Id, DisplayName FROM stackoverflow.Users LIMIT 1;",
        consistency_level=ConsistencyLevel.QUORUM,
    )
)
result_one.one(), result_quorum.one()

### EJERCICIO 5: coste de la consistencia

Con un clúster de un solo nodo, `ONE` y `QUORUM` dan el mismo resultado. Para un clúster de **tres nodos con factor de replicación 3**, calcula cuántas réplicas debe contactar cada nivel (`ONE`, `QUORUM`, `ALL`), tanto en lectura como en escritura, y guarda los seis valores en el diccionario `replicas_necesarias` con claves `'ONE_lectura'`, `'ONE_escritura'`, `'QUORUM_lectura'`, `'QUORUM_escritura'`, `'ALL_lectura'` y `'ALL_escritura'`. Explica además por qué combinar `QUORUM` en lectura y en escritura garantiza ver siempre la última escritura confirmada.

In [ ]:
# Comprobación del EJERCICIO 5: coste de la consistencia
import hashlib

assert "replicas_necesarias" in globals(), (
    "Guarda los resultados en el diccionario replicas_necesarias"
)
claves: list[str] = [
    "ONE_lectura",
    "ONE_escritura",
    "QUORUM_lectura",
    "QUORUM_escritura",
    "ALL_lectura",
    "ALL_escritura",
]
assert sorted(replicas_necesarias) == sorted(claves), f"Las claves deben ser exactamente {claves}"
print(f"Tuyo: {replicas_necesarias}")

# Ningún nivel contacta con menos de una réplica ni con más de las RF = 3 que hay.
assert all(isinstance(v, int) and 1 <= v <= 3 for v in replicas_necesarias.values()), (
    "Cada nivel contacta con entre 1 y RF = 3 réplicas"
)

# La razón de que QUORUM garantice la última escritura no es el valor en sí,
# sino que lectura y escritura se solapen en al menos una réplica.
suma_quorum: int = replicas_necesarias["QUORUM_lectura"] + replicas_necesarias["QUORUM_escritura"]
assert suma_quorum > 3, (
    "QUORUM en lectura + QUORUM en escritura debe superar RF=3 para garantizar solape"
)

# Los seis valores se comparan con el SHA-256 de la respuesta correcta, no con la
# respuesta: la celda no la contiene, y el aviso no dice qué valor falla.
SHA256_REPLICAS: str = "c11440ee3da6ffdd5bdaab3faf963e6eb4b8089558597112099da91656ba6fdd"
clave_replicas: str = "bdge-cassandra-sesion5-ejercicio5:" + ",".join(
    str(replicas_necesarias[clave]) for clave in claves
)
assert hashlib.sha256(clave_replicas.encode("utf-8")).hexdigest() == SHA256_REPLICAS, (
    "Alguno de los seis valores no es el correcto: repasa cuántas réplicas pide cada nivel"
)
print("EJERCICIO 5: comprobado")

### El tipo `COUNTER`

Una columna `counter` sólo admite incrementos y decrementos (`SET visitas = visitas + 1`), nunca una asignación. Cassandra replica esos incrementos de otra forma que las escrituras normales, que se resuelven por marca temporal, así que una tabla con una columna `counter` sólo puede tener otras columnas `counter` además de la clave primaria: los contadores van en una tabla aparte. Tampoco admiten `INSERT` ni nivel de consistencia `ANY`, y reintentar un incremento fallido puede contarlo dos veces.

In [ ]:
# Se vacía tras crearla: sin el TRUNCATE, cada reejecución seguiría sumando
# visitas al mismo contador.
session.execute("""
    CREATE TABLE IF NOT EXISTS stackoverflow.contador_visitas (
        post_id bigint PRIMARY KEY,
        visitas counter
    );
""")
session.execute("TRUNCATE stackoverflow.contador_visitas")
session.execute(
    "UPDATE stackoverflow.contador_visitas SET visitas = visitas + 1 WHERE post_id = 1003;"
)
session.execute(
    "UPDATE stackoverflow.contador_visitas SET visitas = visitas + 1 WHERE post_id = 1003;"
)
session.execute("SELECT * FROM stackoverflow.contador_visitas WHERE post_id = 1003;").one()

In [ ]:
# @title Diapositiva: Ejercicio de diseño: una tabla por consulta
from IPython.display import HTML, display; display(HTML('<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#fff9db;color:#29261d;border:1px solid #dfcc78;border-top:10px solid #c69200;border-radius:10px;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:22px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#c69200" size="8" noshade style="display:none"><div style="color:#c69200;font-size:15px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#c69200" style="color:inherit"><b>A continuación</b></font></div><h1 style="color:#7b2c19;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:34px;line-height:1.15;margin:0 0 22px">Ejercicio de diseño: una tabla por consulta</h1><ul style=\'margin:18px 0 8px;padding-left:30px\'><li style=\'margin:10px 0;list-style-position:outside\'>Las preguntas de un usuario, sin índice ni <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>ALLOW FILTERING</code></li><li style=\'margin:10px 0;list-style-position:outside\'>Una tabla particionada por el autor, con sus preguntas en un <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>set</code></li><li style=\'margin:10px 0;list-style-position:outside\'>Poblarla desde <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>Questions</code> sin escribir <em>tombstones</em></li></ul><blockquote style=\'background:#fff0ae;color:inherit;border:none;border-left:6px solid #c69200;margin:22px 0 0;padding:12px 16px\'><div style=\'margin-top:0px\'>Una consulta frecuente por otra columna pide otra tabla, no un índice</div></blockquote><hr color="#c69200" size="2" noshade style="display:none"></div>'))

### EJERCICIO 6: `QuestionsByUser` con un `SET`

Para que el cuaderno se pueda reejecutar entero, empieza con `DROP TABLE IF EXISTS stackoverflow.QuestionsByUser`.

Crea la tabla `QuestionsByUser` con clave primaria `owner_user_id` y una columna `question_ids set<bigint>`, y puéblala a partir de `Questions` (columna `OwnerUserId`), añadiendo cada `Id` de pregunta al conjunto de su autor.

In [ ]:
# Comprobación del EJERCICIO 6: QuestionsByUser con un SET
from typing import Any

from cassandra.query import SimpleStatement

# El usuario de muestra sale de QuestionsWithUserNames, que sólo guarda las preguntas
# anteriores a FIN_MUESTRA: así tiene al menos una pregunta en las dos tablas.
muestra_pregunta: Any = session.execute(
    "SELECT user_name, owner_user_id FROM stackoverflow.QuestionsWithUserNames LIMIT 1;"
).one()
assert muestra_pregunta is not None, "QuestionsWithUserNames está vacía"
usuario_id: int = muestra_pregunta.owner_user_id
muestra_usuario: Any = session.execute(
    "SELECT question_ids FROM stackoverflow.QuestionsByUser WHERE owner_user_id = %s;",
    (usuario_id,),
).one()
assert muestra_usuario is not None, f"QuestionsByUser no tiene al usuario {usuario_id}"
guardados: set = muestra_usuario.question_ids or set()

# Camino paralelo, por otra tabla: en la partición de QuestionsWithUserNames del
# nombre del usuario, las preguntas con su identificador (varios usuarios pueden
# compartir nombre). Los autores que ya no están en Users se guardaron con el nombre
# «(borrado)», que es también el que tiene la fila de muestra.
esperados_rows: ResultSet = session.execute(
    SimpleStatement(
        "SELECT question_id, owner_user_id FROM stackoverflow.QuestionsWithUserNames "
        "WHERE user_name = %s;",
        fetch_size=2000,
    ),
    (muestra_pregunta.user_name,),
)
esperados: set[int] = {row.question_id for row in esperados_rows if row.owner_user_id == usuario_id}

# QuestionsByUser guarda todas sus preguntas y QuestionsWithUserNames, sólo las
# anteriores a FIN_MUESTRA: se comparan éstas, con la fecha de cada una leída en
# Questions por su clave de partición. Se comparan como conjuntos, sin depender del orden.
guardados_en_muestra: set[int] = set()
for question_id in guardados:
    pregunta: Any = session.execute(
        "SELECT CreationDate FROM stackoverflow.Questions WHERE Id = %s;", (question_id,)
    ).one()
    assert pregunta is not None, f"La pregunta {question_id} no está en Questions"
    if pregunta.creationdate < FIN_MUESTRA:
        guardados_en_muestra.add(question_id)

print(f"Usuario de muestra: {usuario_id}")
print(
    f"IDs en QuestionsByUser: {len(guardados)}, {len(guardados_en_muestra)} anteriores a "
    f"{FIN_MUESTRA:%Y-%m-%d} | IDs en QuestionsWithUserNames: {len(esperados)}"
)
assert esperados, "El usuario de muestra no tiene preguntas en QuestionsWithUserNames"
assert guardados_en_muestra == esperados, (
    "El conjunto de IDs no coincide con las preguntas del usuario"
)
print("EJERCICIO 6: comprobado")

In [ ]:
# @title Diapositiva: Lo que deja la sesión 5
from IPython.display import HTML, display; display(HTML('<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#fff9db;color:#29261d;border:1px solid #dfcc78;border-top:10px solid #287681;border-radius:10px;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:22px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#287681" size="8" noshade style="display:none"><div style="color:#287681;font-size:15px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#287681" style="color:inherit"><b>Recapitulación</b></font></div><h1 style="color:#7b2c19;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:34px;line-height:1.15;margin:0 0 22px">Lo que deja la sesión 5</h1><ul style=\'margin:18px 0 8px;padding-left:30px\'><li style=\'margin:10px 0;list-style-position:outside\'>El <em>driver</em> descubre la topología; el perfil fija balanceo y consistencia</li><li style=\'margin:10px 0;list-style-position:outside\'>La clave de partición decide dónde se busca; el <em>clustering</em>, el orden dentro de la partición</li><li style=\'margin:10px 0;list-style-position:outside\'>Un <em>statement</em> preparado fija forma y tipos, y permite encaminar la petición a su réplica</li><li style=\'margin:10px 0;list-style-position:outside\'>Ni un índice SAI ni <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>ALLOW FILTERING</code> sustituyen a una tabla pensada para la consulta</li><li style=\'margin:10px 0;list-style-position:outside\'>Sin <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>JOIN</code>: se desnormaliza, y el coste pasa a las escrituras</li></ul><blockquote style=\'background:#fff0ae;color:inherit;border:none;border-left:6px solid #287681;margin:22px 0 0;padding:12px 16px\'><div style=\'margin-top:0px\'>En Cassandra modelamos la consulta antes que la tabla</div></blockquote><hr color="#287681" size="2" noshade style="display:none"></div>'))

In [ ]:
# @title Diapositiva: Sesión 6: Cassandra (II)
from IPython.display import HTML, display; display(HTML('<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#fff9db;color:#29261d;border:1px solid #dfcc78;border-top:10px solid #c69200;border-radius:10px;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:22px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#c69200" size="8" noshade style="display:none"><div style="color:#c69200;font-size:15px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#c69200" style="color:inherit"><b>A continuación</b></font></div><h1 style="color:#7b2c19;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:34px;line-height:1.15;margin:0 0 22px">Sesión 6: Cassandra (II)</h1><ul style=\'margin:18px 0 8px;padding-left:30px\'><li style=\'margin:10px 0;list-style-position:outside\'><code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>GROUP BY</code>: agregar sólo dentro de una partición</li><li style=\'margin:10px 0;list-style-position:outside\'>Un índice SAI frente a una tabla específica</li><li style=\'margin:10px 0;list-style-position:outside\'>TTL, marcas temporales, escrituras condicionales y <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>BATCH</code></li><li style=\'margin:10px 0;list-style-position:outside\'>El boletín evaluable de Cassandra</li></ul><blockquote style=\'background:#fff0ae;color:inherit;border:none;border-left:6px solid #c69200;margin:22px 0 0;padding:12px 16px\'><div style=\'margin-top:0px\'>Cada optimización cambia las garantías y el coste operativo</div></blockquote><hr color="#c69200" size="2" noshade style="display:none"></div>'))

## Referencias

- [Documentación oficial de Apache Cassandra 5.0](https://cassandra.apache.org/doc/5.0/index.html)
- [Referencia de CQL](https://cassandra.apache.org/doc/5.0/cassandra/developing/cql/index.html)
- [Modelado de datos en Cassandra](https://cassandra.apache.org/doc/5.0/cassandra/developing/data-modeling/index.html)
- [Compresión de tablas](https://cassandra.apache.org/doc/5.0/cassandra/managing/operating/compression.html)
- [Índices SAI](https://cassandra.apache.org/doc/5.0/cassandra/developing/cql/indexing/sai/sai-overview.html)
- [Documentación del *driver* de Python](https://docs.datastax.com/en/developer/python-driver/latest/), en particular los [perfiles de ejecución](https://docs.datastax.com/en/developer/python-driver/latest/execution_profiles/) y la [paginación](https://docs.datastax.com/en/developer/python-driver/latest/query_paging/)
- [Repositorio del *driver* de Python](https://github.com/apache/cassandra-python-driver), hoy un proyecto de la Apache Software Foundation

## Guía de estudio para el examen

Antes del examen debes ser capaz de hacer esto **sin IA y sin este cuaderno delante**, usando el *keyspace* `stackoverflow` (`Tags`, `Questions`, `Answers`, `Users`):

- Explicar qué decide la clave de partición y qué decide el *clustering*, y por qué ambas cosas juntas forman la clave primaria.
- Escribir un `CREATE TABLE` con clave compuesta y `CLUSTERING ORDER BY`, y justificar por qué esa tabla sirve a una consulta concreta y no a otra.
- Usar un *statement* preparado (`session.prepare()` + `session.execute(stmt, params)`) y explicar por qué es mejor que construir CQL con f-strings.
- Crear un índice SAI, decir qué consultas admite (igualdad y rangos numéricos) y qué sigue sin resolver: es local a cada nodo y no ordena.
- Explicar por qué `execute()` sobre un `CREATE INDEX` no garantiza que el índice ya esté construido, y cómo comprobarlo sin adivinar un tiempo de espera.
- Dada una tabla y su clave primaria, decir qué consultas admite sin `ALLOW FILTERING` (la clave de partición completa y, dentro de ella, las columnas de *clustering* en orden) y cuáles no.
- Decir cuándo hace falta `ALLOW FILTERING`, por qué en general no es recomendable y qué alternativa de diseño la evita.
- Explicar por qué Cassandra no tiene `JOIN` y cómo se resuelve con una tabla desnormalizada, incluyendo qué implica esa duplicación para las escrituras.
- Poblar una tabla derivada a partir de otra con `populate_table()` y explicar qué hace por dentro: escritura concurrente con un *statement* preparado, recorrido del origen por páginas y elección del perfil de ejecución.
- Usar columnas `set`, `list` y `map`, y decir qué diferencia a un `set` de una `list` al añadir un valor repetido.
- Explicar qué nivel de consistencia (`ONE`, `QUORUM`, ...) hace falta en lectura y en escritura para garantizar ver siempre el dato más reciente.
- Explicar por qué una tabla con `COUNTER` no puede mezclar esa columna con columnas normales.

Si al intentar cualquiera de estos puntos te quedas en blanco, vuelve a la parte o al ejercicio correspondiente de este cuaderno antes de pasar a la sesión 6.